### GJ 504 data reduction code
based on the Mathilde's jupyter
Do the stage 1 and 2 of JWST pipeline, with the "new" photometric calibration (aug 2023). No flat field applied

In [1]:
import numpy as np
import glob, os
from astropy.io import fits
from tqdm.notebook import tqdm as tqdm
from scipy import ndimage
import matplotlib.pyplot as plt

#solution tempo du notebook MRS
import multiprocessing
multiprocessing.set_start_method('fork')
from multiprocessing import Pool
# Set the maximum number of processes to spawn based on available cores
usage = 'none' # Either 'none' (single thread), 'quarter', 'half', or 'all' available cores
maxp = 1
print('We will use '+str(maxp)+' CPUs in this run')



import jwst
#from mm_miri.corono import *  #custom code from Mathilde
from photutils.aperture import CircularAperture
from jwst.pipeline import Detector1Pipeline
from jwst.associations import asn_from_list as afl # Tools for creating association files


#import from the mm_miri.corono 
import numpy as np
from jwst.pipeline import Image2Pipeline
from jwst.associations.lib.member import Member
from jwst.associations.asn_from_list import asn_from_list
from jwst.associations.lib.rules_level2_base import DMSLevel2bBase
import scipy.optimize as optimize
from numpy.linalg import eig
import matplotlib.pyplot as plt
from photutils.aperture import CircularAperture






We will use 1 CPUs in this run


In [2]:
#PIPELINE FUNCTIONS : stage 1 & 2

def detector1(filenames):
    """Detector 1 Pipeline stage"""
    det1 = Detector1Pipeline()
    det1.output_dir = det1_dir
    det1.ramp_fit.max_cores = 4
    det1.ipc.skip = True
    det1.jump.rejection_threshold = 8.0 #(ERS Carter et al. 2023 : 8 ; ERO : 5)
    det1.jump.max_cores = 4
    #det1.ramp_fit.save_opt = True #to save the ramp
    det1.save_results = True
    print(filenames)
    det1(filenames)
    return det1

# Define a function that can oversee the multiprocessing
# We'll just pass this a function name and a set of inputs, and it'll figure out the
# multithreading accordingly.
def runmany(step, filenames):
    if __name__ == '__main__':
        p = Pool(maxp)
        res = p.map(step, filenames)
        p.close()
        p.join()


def image2(filenames, image2_dir):
    """Image 2 Pipeline stage"""
    image2 = Image2Pipeline()
    image2.output_dir = image2_dir
    image2.bkg_subtract.skip = False #Subtract background : mean of the two exposures
    image2.assign_wcs.skip = False
    image2.flat_field.skip = True # skip = True because it increases the glowstick 
    image2.photom.skip = False # Photometric calibration (ok after crds 1130) : skip = True to keep the data in DN
    image2.save_results = True
    image2(filenames)
    return image2

def image2_nobkg(filenames, image2_dir):
    """Image 2 Pipeline stage"""
    image2 = Image2Pipeline()
    image2.output_dir = image2_dir
    image2.bkg_subtract.skip = True #Soustraction du background : oui  
    image2.flat_field.skip = True #à ne pas utiliser car augmente le glowstick
    image2.photom.skip = False # Calibration photometrique
    image2.save_results = True
    image2(filenames)
    return image2

def writeasnfile(files, files_bkg, name):
    
    # setup an empty level 2 association structure
    asn = asn_from_list(files, rule=DMSLevel2bBase)
    asn.data['products'] = None

    # set the association name
    asn_name = name

    # set some metadata
    asn['asn_pool'] = asn_name + '_pool'
    asn['asn_type'] = 'image2'

    for n, sci in enumerate(files):
        asn.new_product('{}_dither{}'.format(asn_name, str(n+1)))
        sci_member = Member({'expname': sci, 'exptype': 'science'})    
        new_members = asn.current_product['members']
        new_members.append(sci_member)

        for bkg in files_bkg:
            bkg_member = Member({'expname': bkg, 'exptype': 'background'})
            new_members.append(bkg_member)

    # print the association and save to file
    name, ser = asn.dump()
    print(ser)

    asn_file = asn_name #+ '_lvl2_asn.json'
    with open(asn_file, 'w') as f:
        f.write(ser)
    
    return asn_file

#2ème manière d'écrire le fichier asn, en considérant tous les fichiers comme des données science
def writeasnfile2(files, name):
    # setup an empty level 2 association structure
    asn = asn_from_list(files, rule=DMSLevel2bBase)
    asn.data['products'] = None

    # set the association name
    asn_name = name

    # set some metadata
    asn['asn_pool'] = asn_name + '_pool'
    asn['asn_type'] = 'image2'

    for n, sci in enumerate(files):
        asn.new_product('{}_dither{}'.format(asn_name, str(n+1)))
        sci_member = Member({'expname': sci, 'exptype': 'science'})    
        new_members = asn.current_product['members']
        new_members.append(sci_member)

    name, ser = asn.dump()
    print(ser)

    asn_file = asn_name + '_lvl2_asn.json'
    with open(asn_file, 'w') as f:
        f.write(ser)
    
    return asn_file


### 1. Paths and useful parameters

11.4 micron

In [3]:
#Define path
path = "/home/cperrot/Work/raw/JWST/Target/GTO/GJ504/"
output_dir = path+"output_with_photom/"
name_pl = 'GJ504'
filter_name = ["F1140C"] #["F1065C","F1140C","F1550C"]#, "F2100W"]
cmaps = ["Blues_r","Greens_r","Oranges_r"]

#Number of the observations : useful to retrieve the data 
#For each filter : science obs, background, ref, and ref background
num_obs = [[14,17]] 

D = 6.57 #Diameter of the telescope
wavelength_filter = [11.40]
nb_filter = len(wavelength_filter)
fwhm = [((wavelength_filter[t]*10**-6)/6.57)*(180*3600/np.pi) for t in range(nb_filter)]
pix_size = 0.11 #arcsec/pixel

#Center corono : DEC, RA 
yxc1065 = [112.00, 120.25]  
yxc1140 = [112.16, 119.77]  
yxc1550 = [113.20, 119.73] 
yxc2330 = [167.88, 144.33]  
center_corono = (yxc1065,yxc1140,yxc1550)


In [4]:
#MAKE NEW DIRECTORIES
for ft in range(nb_filter):
   
    #To add the reduced data in new directories with the same observation numbers:
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    if not os.path.exists(output_dir+"obs"+str(num_obs[ft][0])+"/"):
        os.makedirs(output_dir+"obs"+str(num_obs[ft][0])+"/")
    if not os.path.exists(output_dir+"obs"+str(num_obs[ft][0])+"/TA/"):
        os.makedirs(output_dir+"obs"+str(num_obs[ft][0])+"/TA/")
        
    if not os.path.exists(output_dir+"obs"+str(num_obs[ft][1])+"/"):
        os.makedirs(output_dir+"obs"+str(num_obs[ft][1])+"/")
    '''
    if not os.path.exists(output_dir+"obs"+str(num_obs[ft][2])+"/"):
        os.makedirs(output_dir+"obs"+str(num_obs[ft][2])+"/")
    if not os.path.exists(output_dir+"obs"+str(num_obs[ft][3])+"/"):   
        os.makedirs(output_dir+"obs"+str(num_obs[ft][3])+"/")
    
    if not os.path.exists(output_dir+"obs"+str(num_obs[ft][2])+"/TA/"):
        os.makedirs(output_dir+"obs"+str(num_obs[ft][2])+"/TA/")
    if not os.path.exists(output_dir+"obs"+str(num_obs[ft][3])+"/TA/"):   
        os.makedirs(output_dir+"obs"+str(num_obs[ft][3])+"/TA/")
    '''

In [5]:
for ft in range(nb_filter):
    
    files_uncal = sorted(glob.glob(path+'obs'+str(num_obs[ft][0])+'*/*_mirimage_uncal.fits'))
    files_bkg_uncal = sorted(glob.glob(path+'obs'+str(num_obs[ft][1])+'*/*_mirimage_uncal.fits'))
    #files_ref_uncal = sorted(glob.glob(path+'obs'+str(num_obs[ft][2])+'*/*_mirimage_uncal.fits'))
    #files_ref_bkg_uncal = sorted(glob.glob(path+'obs'+str(num_obs[ft][3])+'*/*_mirimage_uncal.fits'))
    files_TA_uncal = sorted(glob.glob(path+'obs'+str(num_obs[ft][0])+'*/TA/*02101*_mirimage_uncal.fits'))
    
    print(files_uncal)
    #To parallelise to get it more efficient ! 
    det1_dir = output_dir+'obs'+str(num_obs[ft][0])+'/'
    runmany(detector1, files_uncal)
    
    print(files_bkg_uncal)
    #To parallelise to get it more efficient ! 
    det1_dir = output_dir+'obs'+str(num_obs[ft][1])+'/'
    runmany(detector1, files_bkg_uncal)
    
    print(files_TA_uncal)
    #To parallelise to get it more efficient ! 
    det1_dir = output_dir+'obs'+str(num_obs[ft][0])+'/TA/'
    runmany(detector1, files_TA_uncal)
    
    #detector1(files_uncal, det1_dir)
    #detector1(files_bkg_uncal)
    #det1_dir = output_dir+'obs'+str(num_obs[ft][0])+'/TA/'
    #detector1(files_TA_uncal)

['/home/cperrot/Work/raw/JWST/Target/GTO/GJ504/obs14/TA/jw01277014001_02101_00001_mirimage_uncal.fits', '/home/cperrot/Work/raw/JWST/Target/GTO/GJ504/obs14/TA/jw01277014001_02101_00002_mirimage_uncal.fits']


2023-11-14 14:44:14,258 - stpipe.Detector1Pipeline - INFO - Detector1Pipeline instance created.
2023-11-14 14:44:14,262 - stpipe.Detector1Pipeline.group_scale - INFO - GroupScaleStep instance created.
2023-11-14 14:44:14,263 - stpipe.Detector1Pipeline.dq_init - INFO - DQInitStep instance created.
2023-11-14 14:44:14,266 - stpipe.Detector1Pipeline.saturation - INFO - SaturationStep instance created.
2023-11-14 14:44:14,268 - stpipe.Detector1Pipeline.ipc - INFO - IPCStep instance created.
2023-11-14 14:44:14,270 - stpipe.Detector1Pipeline.superbias - INFO - SuperBiasStep instance created.
2023-11-14 14:44:14,272 - stpipe.Detector1Pipeline.refpix - INFO - RefPixStep instance created.
2023-11-14 14:44:14,274 - stpipe.Detector1Pipeline.rscd - INFO - RscdStep instance created.
2023-11-14 14:44:14,277 - stpipe.Detector1Pipeline.firstframe - INFO - FirstFrameStep instance created.
2023-11-14 14:44:14,279 - stpipe.Detector1Pipeline.lastframe - INFO - LastFrameStep instance created.
2023-11-14 1

/home/cperrot/Work/raw/JWST/Target/GTO/GJ504/obs14/TA/jw01277014001_02101_00001_mirimage_uncal.fits


2023-11-14 14:44:14,401 - stpipe.Detector1Pipeline - INFO - Step Detector1Pipeline running with args ('/home/cperrot/Work/raw/JWST/Target/GTO/GJ504/obs14/TA/jw01277014001_02101_00001_mirimage_uncal.fits',).
2023-11-14 14:44:14,414 - stpipe.Detector1Pipeline - INFO - Step Detector1Pipeline parameters are: {'pre_hooks': [], 'post_hooks': [], 'output_file': None, 'output_dir': '/home/cperrot/Work/raw/JWST/Target/GTO/GJ504/output_with_photom/obs14/TA/', 'output_ext': '.fits', 'output_use_model': False, 'output_use_index': True, 'save_results': True, 'skip': False, 'suffix': None, 'search_output_file': True, 'input_dir': '', 'save_calibrated_ramp': False, 'steps': {'group_scale': {'pre_hooks': [], 'post_hooks': [], 'output_file': None, 'output_dir': None, 'output_ext': '.fits', 'output_use_model': False, 'output_use_index': True, 'save_results': False, 'skip': False, 'suffix': None, 'search_output_file': True, 'input_dir': ''}, 'dq_init': {'pre_hooks': [], 'post_hooks': [], 'output_file': N

2023-11-14 14:44:16,507 - stpipe.Detector1Pipeline - INFO - Prefetch for SATURATION reference file is '/home/cperrot/crds_cache/references/jwst/miri/jwst_miri_saturation_0034.fits'.
2023-11-14 14:44:16,508 - stpipe.Detector1Pipeline - INFO - Prefetch for SUPERBIAS reference file is 'N/A'.
2023-11-14 14:44:16,510 - stpipe.Detector1Pipeline - INFO - Prefetch for TRAPDENSITY reference file is 'N/A'.
2023-11-14 14:44:16,511 - stpipe.Detector1Pipeline - INFO - Prefetch for TRAPPARS reference file is 'N/A'.
2023-11-14 14:44:16,512 - stpipe.Detector1Pipeline - INFO - Starting calwebb_detector1 ...
2023-11-14 14:44:17,099 - stpipe.Detector1Pipeline.group_scale - INFO - Step group_scale running with args (<RampModel(1, 44, 224, 288) from jw01277014001_02101_00001_mirimage_uncal.fits>,).
2023-11-14 14:44:17,101 - stpipe.Detector1Pipeline.group_scale - INFO - Step group_scale parameters are: {'pre_hooks': [], 'post_hooks': [], 'output_file': None, 'output_dir': None, 'output_ext': '.fits', 'outpu

...

/home/cperrot/Work/raw/JWST/Target/GTO/GJ504/obs14/TA/jw01277014001_02101_00002_mirimage_uncal.fits


2023-11-14 14:44:24,479 - stpipe.Detector1Pipeline - INFO - Step Detector1Pipeline running with args ('/home/cperrot/Work/raw/JWST/Target/GTO/GJ504/obs14/TA/jw01277014001_02101_00002_mirimage_uncal.fits',).
2023-11-14 14:44:24,491 - stpipe.Detector1Pipeline - INFO - Step Detector1Pipeline parameters are: {'pre_hooks': [], 'post_hooks': [], 'output_file': None, 'output_dir': '/home/cperrot/Work/raw/JWST/Target/GTO/GJ504/output_with_photom/obs14/TA/', 'output_ext': '.fits', 'output_use_model': False, 'output_use_index': True, 'save_results': True, 'skip': False, 'suffix': None, 'search_output_file': True, 'input_dir': '', 'save_calibrated_ramp': False, 'steps': {'group_scale': {'pre_hooks': [], 'post_hooks': [], 'output_file': None, 'output_dir': None, 'output_ext': '.fits', 'output_use_model': False, 'output_use_index': True, 'save_results': False, 'skip': False, 'suffix': None, 'search_output_file': True, 'input_dir': ''}, 'dq_init': {'pre_hooks': [], 'post_hooks': [], 'output_file': N

2023-11-14 14:44:24,847 - stpipe.Detector1Pipeline - INFO - Prefetch for SATURATION reference file is '/home/cperrot/crds_cache/references/jwst/miri/jwst_miri_saturation_0034.fits'.
2023-11-14 14:44:24,848 - stpipe.Detector1Pipeline - INFO - Prefetch for SUPERBIAS reference file is 'N/A'.
2023-11-14 14:44:24,848 - stpipe.Detector1Pipeline - INFO - Prefetch for TRAPDENSITY reference file is 'N/A'.
2023-11-14 14:44:24,849 - stpipe.Detector1Pipeline - INFO - Prefetch for TRAPPARS reference file is 'N/A'.
2023-11-14 14:44:24,851 - stpipe.Detector1Pipeline - INFO - Starting calwebb_detector1 ...
2023-11-14 14:44:25,228 - stpipe.Detector1Pipeline.group_scale - INFO - Step group_scale running with args (<RampModel(1, 44, 224, 288) from jw01277014001_02101_00002_mirimage_uncal.fits>,).
2023-11-14 14:44:25,230 - stpipe.Detector1Pipeline.group_scale - INFO - Step group_scale parameters are: {'pre_hooks': [], 'post_hooks': [], 'output_file': None, 'output_dir': None, 'output_ext': '.fits', 'outpu

2023-11-14 14:44:26,720 - stpipe.Detector1Pipeline.linearity - WARNING - Keyword RESERVED_4 does not correspond to an existing DQ mnemonic, so will be ignored
2023-11-14 14:44:26,746 - stpipe.Detector1Pipeline.linearity - WARNING - Keyword UNRELIABLE_RESET does not correspond to an existing DQ mnemonic, so will be ignored
2023-11-14 14:44:26,845 - stpipe.Detector1Pipeline.linearity - INFO - Step linearity done
2023-11-14 14:44:26,935 - stpipe.Detector1Pipeline.rscd - INFO - Step rscd running with args (<RampModel(1, 44, 224, 288) from jw01277014001_02101_00002_mirimage_uncal.fits>,).
2023-11-14 14:44:26,937 - stpipe.Detector1Pipeline.rscd - INFO - Step rscd parameters are: {'pre_hooks': [], 'post_hooks': [], 'output_file': None, 'output_dir': None, 'output_ext': '.fits', 'output_use_model': False, 'output_use_index': True, 'save_results': False, 'skip': False, 'suffix': None, 'search_output_file': True, 'input_dir': '/home/cperrot/Work/raw/JWST/Target/GTO/GJ504/obs14/TA', 'type': 'base

2023-11-14 14:44:28,487 - stpipe.Detector1Pipeline.ramp_fit - INFO - Using GAIN reference file: /home/cperrot/crds_cache/references/jwst/miri/jwst_miri_gain_0021.fits
2023-11-14 14:44:28,557 - stpipe.Detector1Pipeline.ramp_fit - INFO - Extracting gain subarray to match science data
2023-11-14 14:44:28,558 - stpipe.Detector1Pipeline.ramp_fit - INFO - Extracting readnoise subarray to match science data
2023-11-14 14:44:28,559 - stpipe.Detector1Pipeline.ramp_fit - INFO - Using algorithm = ols
2023-11-14 14:44:28,560 - stpipe.Detector1Pipeline.ramp_fit - INFO - Using weighting = optimal
2023-11-14 14:44:28,619 - stpipe.Detector1Pipeline.ramp_fit - INFO - Number of leading groups that are flagged as DO_NOT_USE: 1
2023-11-14 14:44:28,620 - stpipe.Detector1Pipeline.ramp_fit - INFO - MIRI dataset has all pixels in the final group flagged as DO_NOT_USE.
2023-11-14 14:44:32,923 - stpipe.Detector1Pipeline.ramp_fit - INFO - Number of groups per integration: 42
2023-11-14 14:44:32,924 - stpipe.Dete

10.65 micron

In [3]:
#Define path
path = "/home/cperrot/Work/raw/JWST/Target/GTO/GJ504/"
output_dir = path+"output_with_photom/"
name_pl = 'GJ504'
filter_name = ["F1065C"] #["F1065C","F1140C","F1550C"]#, "F2100W"]
cmaps = ["Blues_r","Greens_r","Oranges_r"]

#Number of the observations : useful to retrieve the data 
#For each filter : science obs, background, ref, and ref background
num_obs = [[13,18]] 

D = 6.57 #Diameter of the telescope
wavelength_filter = [10.65]
nb_filter = len(wavelength_filter)
fwhm = [((wavelength_filter[t]*10**-6)/6.57)*(180*3600/np.pi) for t in range(nb_filter)]
pix_size = 0.11 #arcsec/pixel

#Center corono : DEC, RA 
yxc1065 = [112.00, 120.25]  
yxc1140 = [112.16, 119.77]  
yxc1550 = [113.20, 119.73] 
yxc2330 = [167.88, 144.33]  
center_corono = (yxc1065,yxc1140,yxc1550)

In [4]:
#MAKE NEW DIRECTORIES
for ft in range(nb_filter):
   
    #To add the reduced data in new directories with the same observation numbers:
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    if not os.path.exists(output_dir+"obs"+str(num_obs[ft][0])+"/"):
        os.makedirs(output_dir+"obs"+str(num_obs[ft][0])+"/")
    if not os.path.exists(output_dir+"obs"+str(num_obs[ft][0])+"/TA/"):
        os.makedirs(output_dir+"obs"+str(num_obs[ft][0])+"/TA/")
        
    if not os.path.exists(output_dir+"obs"+str(num_obs[ft][1])+"/"):
        os.makedirs(output_dir+"obs"+str(num_obs[ft][1])+"/")
    '''
    if not os.path.exists(output_dir+"obs"+str(num_obs[ft][2])+"/"):
        os.makedirs(output_dir+"obs"+str(num_obs[ft][2])+"/")
    if not os.path.exists(output_dir+"obs"+str(num_obs[ft][3])+"/"):   
        os.makedirs(output_dir+"obs"+str(num_obs[ft][3])+"/")
    
    if not os.path.exists(output_dir+"obs"+str(num_obs[ft][2])+"/TA/"):
        os.makedirs(output_dir+"obs"+str(num_obs[ft][2])+"/TA/")
    if not os.path.exists(output_dir+"obs"+str(num_obs[ft][3])+"/TA/"):   
        os.makedirs(output_dir+"obs"+str(num_obs[ft][3])+"/TA/")
    '''

In [5]:
for ft in range(nb_filter):
    
    files_uncal = sorted(glob.glob(path+'obs'+str(num_obs[ft][0])+'*/*_mirimage_uncal.fits'))
    files_bkg_uncal = sorted(glob.glob(path+'obs'+str(num_obs[ft][1])+'*/*_mirimage_uncal.fits'))
    #files_ref_uncal = sorted(glob.glob(path+'obs'+str(num_obs[ft][2])+'*/*_mirimage_uncal.fits'))
    #files_ref_bkg_uncal = sorted(glob.glob(path+'obs'+str(num_obs[ft][3])+'*/*_mirimage_uncal.fits'))
    files_TA_uncal = sorted(glob.glob(path+'obs'+str(num_obs[ft][0])+'*/TA/*02101*_mirimage_uncal.fits'))
    
    print(files_uncal)
    #To parallelise to get it more efficient ! 
    det1_dir = output_dir+'obs'+str(num_obs[ft][0])+'/'
    runmany(detector1, files_uncal)
    
    print(files_bkg_uncal)
    #To parallelise to get it more efficient ! 
    det1_dir = output_dir+'obs'+str(num_obs[ft][1])+'/'
    runmany(detector1, files_bkg_uncal)
    
    print(files_TA_uncal)
    #To parallelise to get it more efficient ! 
    det1_dir = output_dir+'obs'+str(num_obs[ft][0])+'/TA/'
    runmany(detector1, files_TA_uncal)
    
    #detector1(files_uncal, det1_dir)
    #detector1(files_bkg_uncal)
    #det1_dir = output_dir+'obs'+str(num_obs[ft][0])+'/TA/'
    #detector1(files_TA_uncal)

['/home/cperrot/Work/raw/JWST/Target/GTO/GJ504/obs13/jw01277013001_04101_00001_mirimage_uncal.fits']


2023-11-14 14:55:02,454 - stpipe.Detector1Pipeline - INFO - Detector1Pipeline instance created.
2023-11-14 14:55:02,457 - stpipe.Detector1Pipeline.group_scale - INFO - GroupScaleStep instance created.
2023-11-14 14:55:02,458 - stpipe.Detector1Pipeline.dq_init - INFO - DQInitStep instance created.
2023-11-14 14:55:02,460 - stpipe.Detector1Pipeline.saturation - INFO - SaturationStep instance created.
2023-11-14 14:55:02,461 - stpipe.Detector1Pipeline.ipc - INFO - IPCStep instance created.
2023-11-14 14:55:02,462 - stpipe.Detector1Pipeline.superbias - INFO - SuperBiasStep instance created.
2023-11-14 14:55:02,464 - stpipe.Detector1Pipeline.refpix - INFO - RefPixStep instance created.
2023-11-14 14:55:02,465 - stpipe.Detector1Pipeline.rscd - INFO - RscdStep instance created.
2023-11-14 14:55:02,467 - stpipe.Detector1Pipeline.firstframe - INFO - FirstFrameStep instance created.
2023-11-14 14:55:02,469 - stpipe.Detector1Pipeline.lastframe - INFO - LastFrameStep instance created.
2023-11-14 1

/home/cperrot/Work/raw/JWST/Target/GTO/GJ504/obs13/jw01277013001_04101_00001_mirimage_uncal.fits


2023-11-14 14:55:02,606 - stpipe.Detector1Pipeline - INFO - Step Detector1Pipeline running with args ('/home/cperrot/Work/raw/JWST/Target/GTO/GJ504/obs13/jw01277013001_04101_00001_mirimage_uncal.fits',).
2023-11-14 14:55:02,620 - stpipe.Detector1Pipeline - INFO - Step Detector1Pipeline parameters are: {'pre_hooks': [], 'post_hooks': [], 'output_file': None, 'output_dir': '/home/cperrot/Work/raw/JWST/Target/GTO/GJ504/output_with_photom/obs13/', 'output_ext': '.fits', 'output_use_model': False, 'output_use_index': True, 'save_results': True, 'skip': False, 'suffix': None, 'search_output_file': True, 'input_dir': '', 'save_calibrated_ramp': False, 'steps': {'group_scale': {'pre_hooks': [], 'post_hooks': [], 'output_file': None, 'output_dir': None, 'output_ext': '.fits', 'output_use_model': False, 'output_use_index': True, 'save_results': False, 'skip': False, 'suffix': None, 'search_output_file': True, 'input_dir': ''}, 'dq_init': {'pre_hooks': [], 'post_hooks': [], 'output_file': None, '

2023-11-14 14:55:05,399 - stpipe.Detector1Pipeline - INFO - Prefetch for SATURATION reference file is '/home/cperrot/crds_cache/references/jwst/miri/jwst_miri_saturation_0034.fits'.
2023-11-14 14:55:05,400 - stpipe.Detector1Pipeline - INFO - Prefetch for SUPERBIAS reference file is 'N/A'.
2023-11-14 14:55:05,400 - stpipe.Detector1Pipeline - INFO - Prefetch for TRAPDENSITY reference file is 'N/A'.
2023-11-14 14:55:05,401 - stpipe.Detector1Pipeline - INFO - Prefetch for TRAPPARS reference file is 'N/A'.
2023-11-14 14:55:05,403 - stpipe.Detector1Pipeline - INFO - Starting calwebb_detector1 ...
2023-11-14 14:55:06,573 - stpipe.Detector1Pipeline.group_scale - INFO - Step group_scale running with args (<RampModel(6, 500, 224, 288) from jw01277013001_04101_00001_mirimage_uncal.fits>,).
2023-11-14 14:55:06,576 - stpipe.Detector1Pipeline.group_scale - INFO - Step group_scale parameters are: {'pre_hooks': [], 'post_hooks': [], 'output_file': None, 'output_dir': None, 'output_ext': '.fits', 'outp

...

/home/cperrot/Work/raw/JWST/Target/GTO/GJ504/obs13/TA/jw01277013001_02101_00002_mirimage_uncal.fits


2023-11-14 16:14:14,299 - stpipe.Detector1Pipeline - INFO - Step Detector1Pipeline running with args ('/home/cperrot/Work/raw/JWST/Target/GTO/GJ504/obs13/TA/jw01277013001_02101_00002_mirimage_uncal.fits',).
2023-11-14 16:14:14,314 - stpipe.Detector1Pipeline - INFO - Step Detector1Pipeline parameters are: {'pre_hooks': [], 'post_hooks': [], 'output_file': None, 'output_dir': '/home/cperrot/Work/raw/JWST/Target/GTO/GJ504/output_with_photom/obs13/TA/', 'output_ext': '.fits', 'output_use_model': False, 'output_use_index': True, 'save_results': True, 'skip': False, 'suffix': None, 'search_output_file': True, 'input_dir': '', 'save_calibrated_ramp': False, 'steps': {'group_scale': {'pre_hooks': [], 'post_hooks': [], 'output_file': None, 'output_dir': None, 'output_ext': '.fits', 'output_use_model': False, 'output_use_index': True, 'save_results': False, 'skip': False, 'suffix': None, 'search_output_file': True, 'input_dir': ''}, 'dq_init': {'pre_hooks': [], 'post_hooks': [], 'output_file': N

2023-11-14 16:14:14,579 - stpipe.Detector1Pipeline - INFO - Prefetch for SATURATION reference file is '/home/cperrot/crds_cache/references/jwst/miri/jwst_miri_saturation_0034.fits'.
2023-11-14 16:14:14,580 - stpipe.Detector1Pipeline - INFO - Prefetch for SUPERBIAS reference file is 'N/A'.
2023-11-14 16:14:14,581 - stpipe.Detector1Pipeline - INFO - Prefetch for TRAPDENSITY reference file is 'N/A'.
2023-11-14 16:14:14,581 - stpipe.Detector1Pipeline - INFO - Prefetch for TRAPPARS reference file is 'N/A'.
2023-11-14 16:14:14,583 - stpipe.Detector1Pipeline - INFO - Starting calwebb_detector1 ...
2023-11-14 16:14:14,903 - stpipe.Detector1Pipeline.group_scale - INFO - Step group_scale running with args (<RampModel(1, 44, 224, 288) from jw01277013001_02101_00002_mirimage_uncal.fits>,).
2023-11-14 16:14:14,905 - stpipe.Detector1Pipeline.group_scale - INFO - Step group_scale parameters are: {'pre_hooks': [], 'post_hooks': [], 'output_file': None, 'output_dir': None, 'output_ext': '.fits', 'outpu

2023-11-14 16:14:16,319 - stpipe.Detector1Pipeline.linearity - WARNING - Keyword RESERVED_4 does not correspond to an existing DQ mnemonic, so will be ignored
2023-11-14 16:14:16,336 - stpipe.Detector1Pipeline.linearity - WARNING - Keyword UNRELIABLE_RESET does not correspond to an existing DQ mnemonic, so will be ignored
2023-11-14 16:14:16,436 - stpipe.Detector1Pipeline.linearity - INFO - Step linearity done
2023-11-14 16:14:16,529 - stpipe.Detector1Pipeline.rscd - INFO - Step rscd running with args (<RampModel(1, 44, 224, 288) from jw01277013001_02101_00002_mirimage_uncal.fits>,).
2023-11-14 16:14:16,532 - stpipe.Detector1Pipeline.rscd - INFO - Step rscd parameters are: {'pre_hooks': [], 'post_hooks': [], 'output_file': None, 'output_dir': None, 'output_ext': '.fits', 'output_use_model': False, 'output_use_index': True, 'save_results': False, 'skip': False, 'suffix': None, 'search_output_file': True, 'input_dir': '/home/cperrot/Work/raw/JWST/Target/GTO/GJ504/obs13/TA', 'type': 'base

2023-11-14 16:14:18,041 - stpipe.Detector1Pipeline.ramp_fit - INFO - Using GAIN reference file: /home/cperrot/crds_cache/references/jwst/miri/jwst_miri_gain_0021.fits
2023-11-14 16:14:18,103 - stpipe.Detector1Pipeline.ramp_fit - INFO - Extracting gain subarray to match science data
2023-11-14 16:14:18,104 - stpipe.Detector1Pipeline.ramp_fit - INFO - Extracting readnoise subarray to match science data
2023-11-14 16:14:18,105 - stpipe.Detector1Pipeline.ramp_fit - INFO - Using algorithm = ols
2023-11-14 16:14:18,106 - stpipe.Detector1Pipeline.ramp_fit - INFO - Using weighting = optimal
2023-11-14 16:14:18,158 - stpipe.Detector1Pipeline.ramp_fit - INFO - Number of leading groups that are flagged as DO_NOT_USE: 1
2023-11-14 16:14:18,160 - stpipe.Detector1Pipeline.ramp_fit - INFO - MIRI dataset has all pixels in the final group flagged as DO_NOT_USE.
2023-11-14 16:14:20,747 - stpipe.Detector1Pipeline.ramp_fit - INFO - Number of groups per integration: 42
2023-11-14 16:14:20,748 - stpipe.Dete

15.50 micron

In [3]:
#Define path
path = "/home/cperrot/Work/raw/JWST/Target/GTO/GJ504/"
output_dir = path+"output_with_photom/"
name_pl = 'GJ504'
filter_name = ["F1550C"] #["F1065C","F1140C","F1550C"]#, "F2100W"]
cmaps = ["Blues_r","Greens_r","Oranges_r"]

#Number of the observations : useful to retrieve the data 
#For each filter : science obs, background, ref, and ref background
num_obs = [[15,16]] 

D = 6.57 #Diameter of the telescope
wavelength_filter = [15.50]
nb_filter = len(wavelength_filter)
fwhm = [((wavelength_filter[t]*10**-6)/6.57)*(180*3600/np.pi) for t in range(nb_filter)]
pix_size = 0.11 #arcsec/pixel

#Center corono : DEC, RA 
yxc1065 = [112.00, 120.25]  
yxc1140 = [112.16, 119.77]  
yxc1550 = [113.20, 119.73] 
yxc2330 = [167.88, 144.33]  
center_corono = (yxc1065,yxc1140,yxc1550)

In [4]:
#MAKE NEW DIRECTORIES
for ft in range(nb_filter):
   
    #To add the reduced data in new directories with the same observation numbers:
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    if not os.path.exists(output_dir+"obs"+str(num_obs[ft][0])+"/"):
        os.makedirs(output_dir+"obs"+str(num_obs[ft][0])+"/")
    if not os.path.exists(output_dir+"obs"+str(num_obs[ft][0])+"/TA/"):
        os.makedirs(output_dir+"obs"+str(num_obs[ft][0])+"/TA/")
        
    if not os.path.exists(output_dir+"obs"+str(num_obs[ft][1])+"/"):
        os.makedirs(output_dir+"obs"+str(num_obs[ft][1])+"/")
    '''
    if not os.path.exists(output_dir+"obs"+str(num_obs[ft][2])+"/"):
        os.makedirs(output_dir+"obs"+str(num_obs[ft][2])+"/")
    if not os.path.exists(output_dir+"obs"+str(num_obs[ft][3])+"/"):   
        os.makedirs(output_dir+"obs"+str(num_obs[ft][3])+"/")
    
    if not os.path.exists(output_dir+"obs"+str(num_obs[ft][2])+"/TA/"):
        os.makedirs(output_dir+"obs"+str(num_obs[ft][2])+"/TA/")
    if not os.path.exists(output_dir+"obs"+str(num_obs[ft][3])+"/TA/"):   
        os.makedirs(output_dir+"obs"+str(num_obs[ft][3])+"/TA/")
    '''

In [5]:
for ft in range(nb_filter):
    
    files_uncal = sorted(glob.glob(path+'obs'+str(num_obs[ft][0])+'*/*_mirimage_uncal.fits'))
    files_bkg_uncal = sorted(glob.glob(path+'obs'+str(num_obs[ft][1])+'*/*_mirimage_uncal.fits'))
    #files_ref_uncal = sorted(glob.glob(path+'obs'+str(num_obs[ft][2])+'*/*_mirimage_uncal.fits'))
    #files_ref_bkg_uncal = sorted(glob.glob(path+'obs'+str(num_obs[ft][3])+'*/*_mirimage_uncal.fits'))
    files_TA_uncal = sorted(glob.glob(path+'obs'+str(num_obs[ft][0])+'*/TA/*02101*_mirimage_uncal.fits'))
    
    print(files_uncal)
    #To parallelise to get it more efficient ! 
    det1_dir = output_dir+'obs'+str(num_obs[ft][0])+'/'
    runmany(detector1, files_uncal)
    
    print(files_bkg_uncal)
    #To parallelise to get it more efficient ! 
    det1_dir = output_dir+'obs'+str(num_obs[ft][1])+'/'
    runmany(detector1, files_bkg_uncal)
    
    print(files_TA_uncal)
    #To parallelise to get it more efficient ! 
    det1_dir = output_dir+'obs'+str(num_obs[ft][0])+'/TA/'
    runmany(detector1, files_TA_uncal)
    
    #detector1(files_uncal, det1_dir)
    #detector1(files_bkg_uncal)
    #det1_dir = output_dir+'obs'+str(num_obs[ft][0])+'/TA/'
    #detector1(files_TA_uncal)

['/home/cperrot/Work/raw/JWST/Target/GTO/GJ504/obs15/jw01277015001_04101_00001_mirimage_uncal.fits']


2023-11-14 17:03:58,047 - stpipe.Detector1Pipeline - INFO - Detector1Pipeline instance created.
2023-11-14 17:03:58,050 - stpipe.Detector1Pipeline.group_scale - INFO - GroupScaleStep instance created.
2023-11-14 17:03:58,052 - stpipe.Detector1Pipeline.dq_init - INFO - DQInitStep instance created.
2023-11-14 17:03:58,054 - stpipe.Detector1Pipeline.saturation - INFO - SaturationStep instance created.
2023-11-14 17:03:58,056 - stpipe.Detector1Pipeline.ipc - INFO - IPCStep instance created.
2023-11-14 17:03:58,059 - stpipe.Detector1Pipeline.superbias - INFO - SuperBiasStep instance created.
2023-11-14 17:03:58,062 - stpipe.Detector1Pipeline.refpix - INFO - RefPixStep instance created.
2023-11-14 17:03:58,064 - stpipe.Detector1Pipeline.rscd - INFO - RscdStep instance created.
2023-11-14 17:03:58,065 - stpipe.Detector1Pipeline.firstframe - INFO - FirstFrameStep instance created.
2023-11-14 17:03:58,067 - stpipe.Detector1Pipeline.lastframe - INFO - LastFrameStep instance created.
2023-11-14 1

/home/cperrot/Work/raw/JWST/Target/GTO/GJ504/obs15/jw01277015001_04101_00001_mirimage_uncal.fits


2023-11-14 17:03:58,200 - stpipe.Detector1Pipeline - INFO - Step Detector1Pipeline running with args ('/home/cperrot/Work/raw/JWST/Target/GTO/GJ504/obs15/jw01277015001_04101_00001_mirimage_uncal.fits',).
2023-11-14 17:03:58,215 - stpipe.Detector1Pipeline - INFO - Step Detector1Pipeline parameters are: {'pre_hooks': [], 'post_hooks': [], 'output_file': None, 'output_dir': '/home/cperrot/Work/raw/JWST/Target/GTO/GJ504/output_with_photom/obs15/', 'output_ext': '.fits', 'output_use_model': False, 'output_use_index': True, 'save_results': True, 'skip': False, 'suffix': None, 'search_output_file': True, 'input_dir': '', 'save_calibrated_ramp': False, 'steps': {'group_scale': {'pre_hooks': [], 'post_hooks': [], 'output_file': None, 'output_dir': None, 'output_ext': '.fits', 'output_use_model': False, 'output_use_index': True, 'save_results': False, 'skip': False, 'suffix': None, 'search_output_file': True, 'input_dir': ''}, 'dq_init': {'pre_hooks': [], 'post_hooks': [], 'output_file': None, '

2023-11-14 17:05:19,137 - stpipe.Detector1Pipeline - INFO - Prefetch for REFPIX reference file is 'N/A'.
2023-11-14 17:05:19,138 - stpipe.Detector1Pipeline - INFO - Prefetch for RESET reference file is '/home/cperrot/crds_cache/references/jwst/miri/jwst_miri_reset_0065.fits'.
2023-11-14 17:05:19,138 - stpipe.Detector1Pipeline - INFO - Prefetch for RSCD reference file is '/home/cperrot/crds_cache/references/jwst/miri/jwst_miri_rscd_0017.fits'.
2023-11-14 17:05:19,139 - stpipe.Detector1Pipeline - INFO - Prefetch for SATURATION reference file is '/home/cperrot/crds_cache/references/jwst/miri/jwst_miri_saturation_0034.fits'.
2023-11-14 17:05:19,139 - stpipe.Detector1Pipeline - INFO - Prefetch for SUPERBIAS reference file is 'N/A'.
2023-11-14 17:05:19,140 - stpipe.Detector1Pipeline - INFO - Prefetch for TRAPDENSITY reference file is 'N/A'.
2023-11-14 17:05:19,141 - stpipe.Detector1Pipeline - INFO - Prefetch for TRAPPARS reference file is 'N/A'.
2023-11-14 17:05:19,143 - stpipe.Detector1Pipe

...

/home/cperrot/Work/raw/JWST/Target/GTO/GJ504/obs15/TA/jw01277015001_02101_00002_mirimage_uncal.fits


2023-11-14 17:45:06,510 - stpipe.Detector1Pipeline - INFO - Step Detector1Pipeline running with args ('/home/cperrot/Work/raw/JWST/Target/GTO/GJ504/obs15/TA/jw01277015001_02101_00002_mirimage_uncal.fits',).
2023-11-14 17:45:06,523 - stpipe.Detector1Pipeline - INFO - Step Detector1Pipeline parameters are: {'pre_hooks': [], 'post_hooks': [], 'output_file': None, 'output_dir': '/home/cperrot/Work/raw/JWST/Target/GTO/GJ504/output_with_photom/obs15/TA/', 'output_ext': '.fits', 'output_use_model': False, 'output_use_index': True, 'save_results': True, 'skip': False, 'suffix': None, 'search_output_file': True, 'input_dir': '', 'save_calibrated_ramp': False, 'steps': {'group_scale': {'pre_hooks': [], 'post_hooks': [], 'output_file': None, 'output_dir': None, 'output_ext': '.fits', 'output_use_model': False, 'output_use_index': True, 'save_results': False, 'skip': False, 'suffix': None, 'search_output_file': True, 'input_dir': ''}, 'dq_init': {'pre_hooks': [], 'post_hooks': [], 'output_file': N

2023-11-14 17:45:06,766 - stpipe.Detector1Pipeline - INFO - Prefetch for SATURATION reference file is '/home/cperrot/crds_cache/references/jwst/miri/jwst_miri_saturation_0034.fits'.
2023-11-14 17:45:06,767 - stpipe.Detector1Pipeline - INFO - Prefetch for SUPERBIAS reference file is 'N/A'.
2023-11-14 17:45:06,768 - stpipe.Detector1Pipeline - INFO - Prefetch for TRAPDENSITY reference file is 'N/A'.
2023-11-14 17:45:06,769 - stpipe.Detector1Pipeline - INFO - Prefetch for TRAPPARS reference file is 'N/A'.
2023-11-14 17:45:06,771 - stpipe.Detector1Pipeline - INFO - Starting calwebb_detector1 ...
2023-11-14 17:45:07,062 - stpipe.Detector1Pipeline.group_scale - INFO - Step group_scale running with args (<RampModel(1, 44, 224, 288) from jw01277015001_02101_00002_mirimage_uncal.fits>,).
2023-11-14 17:45:07,064 - stpipe.Detector1Pipeline.group_scale - INFO - Step group_scale parameters are: {'pre_hooks': [], 'post_hooks': [], 'output_file': None, 'output_dir': None, 'output_ext': '.fits', 'outpu

2023-11-14 17:45:08,398 - stpipe.Detector1Pipeline.linearity - WARNING - Keyword RESERVED_4 does not correspond to an existing DQ mnemonic, so will be ignored
2023-11-14 17:45:08,413 - stpipe.Detector1Pipeline.linearity - WARNING - Keyword UNRELIABLE_RESET does not correspond to an existing DQ mnemonic, so will be ignored
2023-11-14 17:45:08,498 - stpipe.Detector1Pipeline.linearity - INFO - Step linearity done
2023-11-14 17:45:08,567 - stpipe.Detector1Pipeline.rscd - INFO - Step rscd running with args (<RampModel(1, 44, 224, 288) from jw01277015001_02101_00002_mirimage_uncal.fits>,).
2023-11-14 17:45:08,569 - stpipe.Detector1Pipeline.rscd - INFO - Step rscd parameters are: {'pre_hooks': [], 'post_hooks': [], 'output_file': None, 'output_dir': None, 'output_ext': '.fits', 'output_use_model': False, 'output_use_index': True, 'save_results': False, 'skip': False, 'suffix': None, 'search_output_file': True, 'input_dir': '/home/cperrot/Work/raw/JWST/Target/GTO/GJ504/obs15/TA', 'type': 'base

2023-11-14 17:45:10,189 - stpipe.Detector1Pipeline.ramp_fit - INFO - Extracting gain subarray to match science data
2023-11-14 17:45:10,191 - stpipe.Detector1Pipeline.ramp_fit - INFO - Extracting readnoise subarray to match science data
2023-11-14 17:45:10,192 - stpipe.Detector1Pipeline.ramp_fit - INFO - Using algorithm = ols
2023-11-14 17:45:10,192 - stpipe.Detector1Pipeline.ramp_fit - INFO - Using weighting = optimal
2023-11-14 17:45:10,244 - stpipe.Detector1Pipeline.ramp_fit - INFO - Number of leading groups that are flagged as DO_NOT_USE: 1
2023-11-14 17:45:10,245 - stpipe.Detector1Pipeline.ramp_fit - INFO - MIRI dataset has all pixels in the final group flagged as DO_NOT_USE.
2023-11-14 17:45:12,844 - stpipe.Detector1Pipeline.ramp_fit - INFO - Number of groups per integration: 42
2023-11-14 17:45:12,846 - stpipe.Detector1Pipeline.ramp_fit - INFO - Number of integrations: 1
2023-11-14 17:45:12,930 - stpipe.Detector1Pipeline.ramp_fit - INFO - Step ramp_fit done
2023-11-14 17:45:13,0

## REFERENCE

HD 95086 reference 10.65 and 11.4 micron

In [6]:
#Define path
path = "/home/cperrot/Work/raw/JWST/Target/GTO/HD95086/"
output_dir = path+"output_with_photom/"
name_pl = 'HD310459'
filter_name = ["F1065C","F1140C"] #["F1065C","F1140C","F1550C"]#, "F2100W"]
cmaps = ["Blues_r","Greens_r","Oranges_r"]

#Number of the observations : useful to retrieve the data 
#For each filter : science obs, background, ref, and ref background
num_obs = [[7,12],[8,11]] 

D = 6.57 #Diameter of the telescope
wavelength_filter = [10.65,11.40]
nb_filter = len(wavelength_filter)
fwhm = [((wavelength_filter[t]*10**-6)/6.57)*(180*3600/np.pi) for t in range(nb_filter)]
pix_size = 0.11 #arcsec/pixel

#Center corono : DEC, RA 
yxc1065 = [112.00, 120.25]  
yxc1140 = [112.16, 119.77]  
yxc1550 = [113.20, 119.73] 
yxc2330 = [167.88, 144.33]  
center_corono = (yxc1065,yxc1140,yxc1550)

In [7]:
#MAKE NEW DIRECTORIES
for ft in range(nb_filter):
   
    #To add the reduced data in new directories with the same observation numbers:
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    if not os.path.exists(output_dir+"obs"+str(num_obs[ft][0])+"/"):
        os.makedirs(output_dir+"obs"+str(num_obs[ft][0])+"/")
    if not os.path.exists(output_dir+"obs"+str(num_obs[ft][0])+"/TA/"):
        os.makedirs(output_dir+"obs"+str(num_obs[ft][0])+"/TA/")
        
    if not os.path.exists(output_dir+"obs"+str(num_obs[ft][1])+"/"):
        os.makedirs(output_dir+"obs"+str(num_obs[ft][1])+"/")
    '''
    if not os.path.exists(output_dir+"obs"+str(num_obs[ft][2])+"/"):
        os.makedirs(output_dir+"obs"+str(num_obs[ft][2])+"/")
    if not os.path.exists(output_dir+"obs"+str(num_obs[ft][3])+"/"):   
        os.makedirs(output_dir+"obs"+str(num_obs[ft][3])+"/")
    
    if not os.path.exists(output_dir+"obs"+str(num_obs[ft][2])+"/TA/"):
        os.makedirs(output_dir+"obs"+str(num_obs[ft][2])+"/TA/")
    if not os.path.exists(output_dir+"obs"+str(num_obs[ft][3])+"/TA/"):   
        os.makedirs(output_dir+"obs"+str(num_obs[ft][3])+"/TA/")
    '''

In [8]:
for ft in range(nb_filter):
    
    files_uncal = sorted(glob.glob(path+'obs'+str(num_obs[ft][0])+'*/*_mirimage_uncal.fits'))
    files_bkg_uncal = sorted(glob.glob(path+'obs'+str(num_obs[ft][1])+'*/*_mirimage_uncal.fits'))
    #files_ref_uncal = sorted(glob.glob(path+'obs'+str(num_obs[ft][2])+'*/*_mirimage_uncal.fits'))
    #files_ref_bkg_uncal = sorted(glob.glob(path+'obs'+str(num_obs[ft][3])+'*/*_mirimage_uncal.fits'))
    files_TA_uncal = sorted(glob.glob(path+'obs'+str(num_obs[ft][0])+'*/TA/*02101*_mirimage_uncal.fits'))
    
    print(files_uncal)
    #To parallelise to get it more efficient ! 
    det1_dir = output_dir+'obs'+str(num_obs[ft][0])+'/'
    runmany(detector1, files_uncal)
    
    print(files_bkg_uncal)
    #To parallelise to get it more efficient ! 
    det1_dir = output_dir+'obs'+str(num_obs[ft][1])+'/'
    runmany(detector1, files_bkg_uncal)
    
    print(files_TA_uncal)
    #To parallelise to get it more efficient ! 
    det1_dir = output_dir+'obs'+str(num_obs[ft][0])+'/TA/'
    runmany(detector1, files_TA_uncal)
    
    #detector1(files_uncal, det1_dir)
    #detector1(files_bkg_uncal)
    #det1_dir = output_dir+'obs'+str(num_obs[ft][0])+'/TA/'
    #detector1(files_TA_uncal)

['/home/cperrot/Work/raw/JWST/Target/GTO/HD95086/obs7/jw01277007001_04101_00001_mirimage_uncal.fits', '/home/cperrot/Work/raw/JWST/Target/GTO/HD95086/obs7/jw01277007001_04101_00002_mirimage_uncal.fits', '/home/cperrot/Work/raw/JWST/Target/GTO/HD95086/obs7/jw01277007001_04101_00003_mirimage_uncal.fits', '/home/cperrot/Work/raw/JWST/Target/GTO/HD95086/obs7/jw01277007001_04101_00004_mirimage_uncal.fits', '/home/cperrot/Work/raw/JWST/Target/GTO/HD95086/obs7/jw01277007001_04101_00005_mirimage_uncal.fits', '/home/cperrot/Work/raw/JWST/Target/GTO/HD95086/obs7/jw01277007001_04101_00006_mirimage_uncal.fits', '/home/cperrot/Work/raw/JWST/Target/GTO/HD95086/obs7/jw01277007001_04101_00007_mirimage_uncal.fits', '/home/cperrot/Work/raw/JWST/Target/GTO/HD95086/obs7/jw01277007001_04101_00008_mirimage_uncal.fits', '/home/cperrot/Work/raw/JWST/Target/GTO/HD95086/obs7/jw01277007001_04101_00009_mirimage_uncal.fits']


2023-11-14 17:45:13,401 - stpipe.Detector1Pipeline - INFO - Detector1Pipeline instance created.
2023-11-14 17:45:13,405 - stpipe.Detector1Pipeline.group_scale - INFO - GroupScaleStep instance created.
2023-11-14 17:45:13,406 - stpipe.Detector1Pipeline.dq_init - INFO - DQInitStep instance created.
2023-11-14 17:45:13,408 - stpipe.Detector1Pipeline.saturation - INFO - SaturationStep instance created.
2023-11-14 17:45:13,410 - stpipe.Detector1Pipeline.ipc - INFO - IPCStep instance created.
2023-11-14 17:45:13,412 - stpipe.Detector1Pipeline.superbias - INFO - SuperBiasStep instance created.
2023-11-14 17:45:13,414 - stpipe.Detector1Pipeline.refpix - INFO - RefPixStep instance created.
2023-11-14 17:45:13,416 - stpipe.Detector1Pipeline.rscd - INFO - RscdStep instance created.
2023-11-14 17:45:13,417 - stpipe.Detector1Pipeline.firstframe - INFO - FirstFrameStep instance created.
2023-11-14 17:45:13,419 - stpipe.Detector1Pipeline.lastframe - INFO - LastFrameStep instance created.
2023-11-14 1

/home/cperrot/Work/raw/JWST/Target/GTO/HD95086/obs7/jw01277007001_04101_00001_mirimage_uncal.fits


2023-11-14 17:45:13,538 - stpipe.Detector1Pipeline - INFO - Step Detector1Pipeline running with args ('/home/cperrot/Work/raw/JWST/Target/GTO/HD95086/obs7/jw01277007001_04101_00001_mirimage_uncal.fits',).
2023-11-14 17:45:13,549 - stpipe.Detector1Pipeline - INFO - Step Detector1Pipeline parameters are: {'pre_hooks': [], 'post_hooks': [], 'output_file': None, 'output_dir': '/home/cperrot/Work/raw/JWST/Target/GTO/HD95086/output_with_photom/obs7/', 'output_ext': '.fits', 'output_use_model': False, 'output_use_index': True, 'save_results': True, 'skip': False, 'suffix': None, 'search_output_file': True, 'input_dir': '', 'save_calibrated_ramp': False, 'steps': {'group_scale': {'pre_hooks': [], 'post_hooks': [], 'output_file': None, 'output_dir': None, 'output_ext': '.fits', 'output_use_model': False, 'output_use_index': True, 'save_results': False, 'skip': False, 'suffix': None, 'search_output_file': True, 'input_dir': ''}, 'dq_init': {'pre_hooks': [], 'post_hooks': [], 'output_file': None,

2023-11-14 17:45:15,603 - stpipe.Detector1Pipeline - INFO - Prefetch for SATURATION reference file is '/home/cperrot/crds_cache/references/jwst/miri/jwst_miri_saturation_0034.fits'.
2023-11-14 17:45:15,604 - stpipe.Detector1Pipeline - INFO - Prefetch for SUPERBIAS reference file is 'N/A'.
2023-11-14 17:45:15,604 - stpipe.Detector1Pipeline - INFO - Prefetch for TRAPDENSITY reference file is 'N/A'.
2023-11-14 17:45:15,605 - stpipe.Detector1Pipeline - INFO - Prefetch for TRAPPARS reference file is 'N/A'.
2023-11-14 17:45:15,607 - stpipe.Detector1Pipeline - INFO - Starting calwebb_detector1 ...
2023-11-14 17:45:16,195 - stpipe.Detector1Pipeline.group_scale - INFO - Step group_scale running with args (<RampModel(9, 150, 224, 288) from jw01277007001_04101_00001_mirimage_uncal.fits>,).
2023-11-14 17:45:16,198 - stpipe.Detector1Pipeline.group_scale - INFO - Step group_scale parameters are: {'pre_hooks': [], 'post_hooks': [], 'output_file': None, 'output_dir': None, 'output_ext': '.fits', 'outp

...

/home/cperrot/Work/raw/JWST/Target/GTO/HD95086/obs8/TA/jw01277008001_02101_00002_mirimage_uncal.fits


2023-11-14 18:44:48,559 - stpipe.Detector1Pipeline - INFO - Step Detector1Pipeline running with args ('/home/cperrot/Work/raw/JWST/Target/GTO/HD95086/obs8/TA/jw01277008001_02101_00002_mirimage_uncal.fits',).
2023-11-14 18:44:48,572 - stpipe.Detector1Pipeline - INFO - Step Detector1Pipeline parameters are: {'pre_hooks': [], 'post_hooks': [], 'output_file': None, 'output_dir': '/home/cperrot/Work/raw/JWST/Target/GTO/HD95086/output_with_photom/obs8/TA/', 'output_ext': '.fits', 'output_use_model': False, 'output_use_index': True, 'save_results': True, 'skip': False, 'suffix': None, 'search_output_file': True, 'input_dir': '', 'save_calibrated_ramp': False, 'steps': {'group_scale': {'pre_hooks': [], 'post_hooks': [], 'output_file': None, 'output_dir': None, 'output_ext': '.fits', 'output_use_model': False, 'output_use_index': True, 'save_results': False, 'skip': False, 'suffix': None, 'search_output_file': True, 'input_dir': ''}, 'dq_init': {'pre_hooks': [], 'post_hooks': [], 'output_file':

2023-11-14 18:44:48,811 - stpipe.Detector1Pipeline - INFO - Prefetch for SATURATION reference file is '/home/cperrot/crds_cache/references/jwst/miri/jwst_miri_saturation_0034.fits'.
2023-11-14 18:44:48,812 - stpipe.Detector1Pipeline - INFO - Prefetch for SUPERBIAS reference file is 'N/A'.
2023-11-14 18:44:48,813 - stpipe.Detector1Pipeline - INFO - Prefetch for TRAPDENSITY reference file is 'N/A'.
2023-11-14 18:44:48,813 - stpipe.Detector1Pipeline - INFO - Prefetch for TRAPPARS reference file is 'N/A'.
2023-11-14 18:44:48,815 - stpipe.Detector1Pipeline - INFO - Starting calwebb_detector1 ...
2023-11-14 18:44:49,107 - stpipe.Detector1Pipeline.group_scale - INFO - Step group_scale running with args (<RampModel(1, 44, 224, 288) from jw01277008001_02101_00002_mirimage_uncal.fits>,).
2023-11-14 18:44:49,110 - stpipe.Detector1Pipeline.group_scale - INFO - Step group_scale parameters are: {'pre_hooks': [], 'post_hooks': [], 'output_file': None, 'output_dir': None, 'output_ext': '.fits', 'outpu

2023-11-14 18:44:50,413 - stpipe.Detector1Pipeline.linearity - WARNING - Keyword RESERVED_4 does not correspond to an existing DQ mnemonic, so will be ignored
2023-11-14 18:44:50,430 - stpipe.Detector1Pipeline.linearity - WARNING - Keyword UNRELIABLE_RESET does not correspond to an existing DQ mnemonic, so will be ignored
2023-11-14 18:44:50,512 - stpipe.Detector1Pipeline.linearity - INFO - Step linearity done
2023-11-14 18:44:50,582 - stpipe.Detector1Pipeline.rscd - INFO - Step rscd running with args (<RampModel(1, 44, 224, 288) from jw01277008001_02101_00002_mirimage_uncal.fits>,).
2023-11-14 18:44:50,584 - stpipe.Detector1Pipeline.rscd - INFO - Step rscd parameters are: {'pre_hooks': [], 'post_hooks': [], 'output_file': None, 'output_dir': None, 'output_ext': '.fits', 'output_use_model': False, 'output_use_index': True, 'save_results': False, 'skip': False, 'suffix': None, 'search_output_file': True, 'input_dir': '/home/cperrot/Work/raw/JWST/Target/GTO/HD95086/obs8/TA', 'type': 'bas

2023-11-14 18:44:51,845 - stpipe.Detector1Pipeline.ramp_fit - INFO - Using READNOISE reference file: /home/cperrot/crds_cache/references/jwst/miri/jwst_miri_readnoise_0085.fits
2023-11-14 18:44:51,847 - stpipe.Detector1Pipeline.ramp_fit - INFO - Using GAIN reference file: /home/cperrot/crds_cache/references/jwst/miri/jwst_miri_gain_0021.fits
2023-11-14 18:44:51,911 - stpipe.Detector1Pipeline.ramp_fit - INFO - Extracting gain subarray to match science data
2023-11-14 18:44:51,912 - stpipe.Detector1Pipeline.ramp_fit - INFO - Extracting readnoise subarray to match science data
2023-11-14 18:44:51,914 - stpipe.Detector1Pipeline.ramp_fit - INFO - Using algorithm = ols
2023-11-14 18:44:51,914 - stpipe.Detector1Pipeline.ramp_fit - INFO - Using weighting = optimal
2023-11-14 18:44:51,966 - stpipe.Detector1Pipeline.ramp_fit - INFO - Number of leading groups that are flagged as DO_NOT_USE: 1
2023-11-14 18:44:51,967 - stpipe.Detector1Pipeline.ramp_fit - INFO - MIRI dataset has all pixels in the f

HD 181318 reference 11.4 micron

In [9]:
#Define path
path = "/home/cperrot/Work/raw/JWST/Target/GO/P3662/"
output_dir = path+"output_with_photom/"
name_pl = 'HD203010'
filter_name = ["F1140C"] #["F1065C","F1140C","F1550C"]#, "F2100W"]
cmaps = ["Blues_r","Greens_r","Oranges_r"]

#Number of the observations : useful to retrieve the data 
#For each filter : science obs, background, ref, and ref background
num_obs = [[9,10]] 

D = 6.57 #Diameter of the telescope
wavelength_filter = [11.40]
nb_filter = len(wavelength_filter)
fwhm = [((wavelength_filter[t]*10**-6)/6.57)*(180*3600/np.pi) for t in range(nb_filter)]
pix_size = 0.11 #arcsec/pixel

#Center corono : DEC, RA 
yxc1065 = [112.00, 120.25]  
yxc1140 = [112.16, 119.77]  
yxc1550 = [113.20, 119.73] 
yxc2330 = [167.88, 144.33]  
center_corono = (yxc1065,yxc1140,yxc1550)

In [10]:
#MAKE NEW DIRECTORIES
for ft in range(nb_filter):
   
    #To add the reduced data in new directories with the same observation numbers:
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    if not os.path.exists(output_dir+"obs"+str(num_obs[ft][0])+"/"):
        os.makedirs(output_dir+"obs"+str(num_obs[ft][0])+"/")
    if not os.path.exists(output_dir+"obs"+str(num_obs[ft][0])+"/TA/"):
        os.makedirs(output_dir+"obs"+str(num_obs[ft][0])+"/TA/")
        
    if not os.path.exists(output_dir+"obs"+str(num_obs[ft][1])+"/"):
        os.makedirs(output_dir+"obs"+str(num_obs[ft][1])+"/")
    '''
    if not os.path.exists(output_dir+"obs"+str(num_obs[ft][2])+"/"):
        os.makedirs(output_dir+"obs"+str(num_obs[ft][2])+"/")
    if not os.path.exists(output_dir+"obs"+str(num_obs[ft][3])+"/"):   
        os.makedirs(output_dir+"obs"+str(num_obs[ft][3])+"/")
    
    if not os.path.exists(output_dir+"obs"+str(num_obs[ft][2])+"/TA/"):
        os.makedirs(output_dir+"obs"+str(num_obs[ft][2])+"/TA/")
    if not os.path.exists(output_dir+"obs"+str(num_obs[ft][3])+"/TA/"):   
        os.makedirs(output_dir+"obs"+str(num_obs[ft][3])+"/TA/")
    '''

In [11]:
for ft in range(nb_filter):
    
    files_uncal = sorted(glob.glob(path+'obs'+str(num_obs[ft][0])+'*/*_mirimage_uncal.fits'))
    files_bkg_uncal = sorted(glob.glob(path+'obs'+str(num_obs[ft][1])+'*/*_mirimage_uncal.fits'))
    #files_ref_uncal = sorted(glob.glob(path+'obs'+str(num_obs[ft][2])+'*/*_mirimage_uncal.fits'))
    #files_ref_bkg_uncal = sorted(glob.glob(path+'obs'+str(num_obs[ft][3])+'*/*_mirimage_uncal.fits'))
    files_TA_uncal = sorted(glob.glob(path+'obs'+str(num_obs[ft][0])+'*/TA/*02101*_mirimage_uncal.fits'))
    
    print(files_uncal)
    #To parallelise to get it more efficient ! 
    det1_dir = output_dir+'obs'+str(num_obs[ft][0])+'/'
    runmany(detector1, files_uncal)
    
    print(files_bkg_uncal)
    #To parallelise to get it more efficient ! 
    det1_dir = output_dir+'obs'+str(num_obs[ft][1])+'/'
    runmany(detector1, files_bkg_uncal)
    
    print(files_TA_uncal)
    #To parallelise to get it more efficient ! 
    det1_dir = output_dir+'obs'+str(num_obs[ft][0])+'/TA/'
    runmany(detector1, files_TA_uncal)
    
    #detector1(files_uncal, det1_dir)
    #detector1(files_bkg_uncal)
    #det1_dir = output_dir+'obs'+str(num_obs[ft][0])+'/TA/'
    #detector1(files_TA_uncal)

['/home/cperrot/Work/raw/JWST/Target/GO/P3662/obs9/jw03662009001_04101_00001_mirimage_uncal.fits', '/home/cperrot/Work/raw/JWST/Target/GO/P3662/obs9/jw03662009001_04101_00002_mirimage_uncal.fits', '/home/cperrot/Work/raw/JWST/Target/GO/P3662/obs9/jw03662009001_04101_00003_mirimage_uncal.fits', '/home/cperrot/Work/raw/JWST/Target/GO/P3662/obs9/jw03662009001_04101_00004_mirimage_uncal.fits', '/home/cperrot/Work/raw/JWST/Target/GO/P3662/obs9/jw03662009001_04101_00005_mirimage_uncal.fits', '/home/cperrot/Work/raw/JWST/Target/GO/P3662/obs9/jw03662009001_04101_00006_mirimage_uncal.fits', '/home/cperrot/Work/raw/JWST/Target/GO/P3662/obs9/jw03662009001_04101_00007_mirimage_uncal.fits', '/home/cperrot/Work/raw/JWST/Target/GO/P3662/obs9/jw03662009001_04101_00008_mirimage_uncal.fits', '/home/cperrot/Work/raw/JWST/Target/GO/P3662/obs9/jw03662009001_04101_00009_mirimage_uncal.fits']


2023-11-14 18:44:54,905 - stpipe.Detector1Pipeline - INFO - Detector1Pipeline instance created.
2023-11-14 18:44:54,908 - stpipe.Detector1Pipeline.group_scale - INFO - GroupScaleStep instance created.
2023-11-14 18:44:54,910 - stpipe.Detector1Pipeline.dq_init - INFO - DQInitStep instance created.
2023-11-14 18:44:54,912 - stpipe.Detector1Pipeline.saturation - INFO - SaturationStep instance created.
2023-11-14 18:44:54,913 - stpipe.Detector1Pipeline.ipc - INFO - IPCStep instance created.
2023-11-14 18:44:54,914 - stpipe.Detector1Pipeline.superbias - INFO - SuperBiasStep instance created.
2023-11-14 18:44:54,916 - stpipe.Detector1Pipeline.refpix - INFO - RefPixStep instance created.
2023-11-14 18:44:54,917 - stpipe.Detector1Pipeline.rscd - INFO - RscdStep instance created.
2023-11-14 18:44:54,919 - stpipe.Detector1Pipeline.firstframe - INFO - FirstFrameStep instance created.
2023-11-14 18:44:54,921 - stpipe.Detector1Pipeline.lastframe - INFO - LastFrameStep instance created.
2023-11-14 1

/home/cperrot/Work/raw/JWST/Target/GO/P3662/obs9/jw03662009001_04101_00001_mirimage_uncal.fits


2023-11-14 18:44:55,063 - stpipe.Detector1Pipeline - INFO - Step Detector1Pipeline running with args ('/home/cperrot/Work/raw/JWST/Target/GO/P3662/obs9/jw03662009001_04101_00001_mirimage_uncal.fits',).
2023-11-14 18:44:55,072 - stpipe.Detector1Pipeline - INFO - Step Detector1Pipeline parameters are: {'pre_hooks': [], 'post_hooks': [], 'output_file': None, 'output_dir': '/home/cperrot/Work/raw/JWST/Target/GO/P3662/output_with_photom/obs9/', 'output_ext': '.fits', 'output_use_model': False, 'output_use_index': True, 'save_results': True, 'skip': False, 'suffix': None, 'search_output_file': True, 'input_dir': '', 'save_calibrated_ramp': False, 'steps': {'group_scale': {'pre_hooks': [], 'post_hooks': [], 'output_file': None, 'output_dir': None, 'output_ext': '.fits', 'output_use_model': False, 'output_use_index': True, 'save_results': False, 'skip': False, 'suffix': None, 'search_output_file': True, 'input_dir': ''}, 'dq_init': {'pre_hooks': [], 'post_hooks': [], 'output_file': None, 'outp

2023-11-14 18:44:56,971 - stpipe.Detector1Pipeline - INFO - Prefetch for SATURATION reference file is '/home/cperrot/crds_cache/references/jwst/miri/jwst_miri_saturation_0034.fits'.
2023-11-14 18:44:56,972 - stpipe.Detector1Pipeline - INFO - Prefetch for SUPERBIAS reference file is 'N/A'.
2023-11-14 18:44:56,973 - stpipe.Detector1Pipeline - INFO - Prefetch for TRAPDENSITY reference file is 'N/A'.
2023-11-14 18:44:56,973 - stpipe.Detector1Pipeline - INFO - Prefetch for TRAPPARS reference file is 'N/A'.
2023-11-14 18:44:56,976 - stpipe.Detector1Pipeline - INFO - Starting calwebb_detector1 ...
2023-11-14 18:44:57,594 - stpipe.Detector1Pipeline.group_scale - INFO - Step group_scale running with args (<RampModel(3, 500, 224, 288) from jw03662009001_04101_00001_mirimage_uncal.fits>,).
2023-11-14 18:44:57,596 - stpipe.Detector1Pipeline.group_scale - INFO - Step group_scale parameters are: {'pre_hooks': [], 'post_hooks': [], 'output_file': None, 'output_dir': None, 'output_ext': '.fits', 'outp

...

/home/cperrot/Work/raw/JWST/Target/GO/P3662/obs9/TA/jw03662009001_02101_00002_mirimage_uncal.fits


2023-11-14 21:11:08,299 - stpipe.Detector1Pipeline - INFO - Step Detector1Pipeline running with args ('/home/cperrot/Work/raw/JWST/Target/GO/P3662/obs9/TA/jw03662009001_02101_00002_mirimage_uncal.fits',).
2023-11-14 21:11:08,310 - stpipe.Detector1Pipeline - INFO - Step Detector1Pipeline parameters are: {'pre_hooks': [], 'post_hooks': [], 'output_file': None, 'output_dir': '/home/cperrot/Work/raw/JWST/Target/GO/P3662/output_with_photom/obs9/TA/', 'output_ext': '.fits', 'output_use_model': False, 'output_use_index': True, 'save_results': True, 'skip': False, 'suffix': None, 'search_output_file': True, 'input_dir': '', 'save_calibrated_ramp': False, 'steps': {'group_scale': {'pre_hooks': [], 'post_hooks': [], 'output_file': None, 'output_dir': None, 'output_ext': '.fits', 'output_use_model': False, 'output_use_index': True, 'save_results': False, 'skip': False, 'suffix': None, 'search_output_file': True, 'input_dir': ''}, 'dq_init': {'pre_hooks': [], 'post_hooks': [], 'output_file': None,

2023-11-14 21:11:10,095 - stpipe.Detector1Pipeline - INFO - Prefetch for SATURATION reference file is '/home/cperrot/crds_cache/references/jwst/miri/jwst_miri_saturation_0034.fits'.
2023-11-14 21:11:10,095 - stpipe.Detector1Pipeline - INFO - Prefetch for SUPERBIAS reference file is 'N/A'.
2023-11-14 21:11:10,096 - stpipe.Detector1Pipeline - INFO - Prefetch for TRAPDENSITY reference file is 'N/A'.
2023-11-14 21:11:10,097 - stpipe.Detector1Pipeline - INFO - Prefetch for TRAPPARS reference file is 'N/A'.
2023-11-14 21:11:10,098 - stpipe.Detector1Pipeline - INFO - Starting calwebb_detector1 ...
2023-11-14 21:11:10,452 - stpipe.Detector1Pipeline.group_scale - INFO - Step group_scale running with args (<RampModel(1, 22, 224, 288) from jw03662009001_02101_00002_mirimage_uncal.fits>,).
2023-11-14 21:11:10,454 - stpipe.Detector1Pipeline.group_scale - INFO - Step group_scale parameters are: {'pre_hooks': [], 'post_hooks': [], 'output_file': None, 'output_dir': None, 'output_ext': '.fits', 'outpu

2023-11-14 21:11:11,717 - stpipe.Detector1Pipeline.linearity - WARNING - Keyword RESERVED_4 does not correspond to an existing DQ mnemonic, so will be ignored
2023-11-14 21:11:11,735 - stpipe.Detector1Pipeline.linearity - WARNING - Keyword UNRELIABLE_RESET does not correspond to an existing DQ mnemonic, so will be ignored
2023-11-14 21:11:11,807 - stpipe.Detector1Pipeline.linearity - INFO - Step linearity done
2023-11-14 21:11:11,875 - stpipe.Detector1Pipeline.rscd - INFO - Step rscd running with args (<RampModel(1, 22, 224, 288) from jw03662009001_02101_00002_mirimage_uncal.fits>,).
2023-11-14 21:11:11,877 - stpipe.Detector1Pipeline.rscd - INFO - Step rscd parameters are: {'pre_hooks': [], 'post_hooks': [], 'output_file': None, 'output_dir': None, 'output_ext': '.fits', 'output_use_model': False, 'output_use_index': True, 'save_results': False, 'skip': False, 'suffix': None, 'search_output_file': True, 'input_dir': '/home/cperrot/Work/raw/JWST/Target/GO/P3662/obs9/TA', 'type': 'baseli

2023-11-14 21:11:13,055 - stpipe.Detector1Pipeline.ramp_fit - INFO - Extracting gain subarray to match science data
2023-11-14 21:11:13,057 - stpipe.Detector1Pipeline.ramp_fit - INFO - Extracting readnoise subarray to match science data
2023-11-14 21:11:13,058 - stpipe.Detector1Pipeline.ramp_fit - INFO - Using algorithm = ols
2023-11-14 21:11:13,059 - stpipe.Detector1Pipeline.ramp_fit - INFO - Using weighting = optimal
2023-11-14 21:11:13,107 - stpipe.Detector1Pipeline.ramp_fit - INFO - Number of leading groups that are flagged as DO_NOT_USE: 1
2023-11-14 21:11:13,109 - stpipe.Detector1Pipeline.ramp_fit - INFO - MIRI dataset has all pixels in the final group flagged as DO_NOT_USE.
2023-11-14 21:11:13,779 - stpipe.Detector1Pipeline.ramp_fit - INFO - Number of groups per integration: 20
2023-11-14 21:11:13,781 - stpipe.Detector1Pipeline.ramp_fit - INFO - Number of integrations: 1
2023-11-14 21:11:13,867 - stpipe.Detector1Pipeline.ramp_fit - INFO - Step ramp_fit done
2023-11-14 21:11:13,9

HR 8799 15.5 mic

In [12]:
#Define path
path = "/home/cperrot/Work/raw/JWST/Target/GTO/HR8799/"
output_dir = path+"output_with_photom/"
name_pl = 'HD218261'
filter_name = ["F1550C"] #["F1065C","F1140C","F1550C"]#, "F2100W"]
cmaps = ["Blues_r","Greens_r","Oranges_r"]

#Number of the observations : useful to retrieve the data 
#For each filter : science obs, background, ref, and ref background
num_obs = [[16,18]] 

D = 6.57 #Diameter of the telescope
wavelength_filter = [15.50]
nb_filter = len(wavelength_filter)
fwhm = [((wavelength_filter[t]*10**-6)/6.57)*(180*3600/np.pi) for t in range(nb_filter)]
pix_size = 0.11 #arcsec/pixel

#Center corono : DEC, RA 
yxc1065 = [112.00, 120.25]  
yxc1140 = [112.16, 119.77]  
yxc1550 = [113.20, 119.73] 
yxc2330 = [167.88, 144.33]  
center_corono = (yxc1065,yxc1140,yxc1550)

In [13]:
#MAKE NEW DIRECTORIES
for ft in range(nb_filter):
   
    #To add the reduced data in new directories with the same observation numbers:
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    if not os.path.exists(output_dir+"obs"+str(num_obs[ft][0])+"/"):
        os.makedirs(output_dir+"obs"+str(num_obs[ft][0])+"/")
    if not os.path.exists(output_dir+"obs"+str(num_obs[ft][0])+"/TA/"):
        os.makedirs(output_dir+"obs"+str(num_obs[ft][0])+"/TA/")
        
    if not os.path.exists(output_dir+"obs"+str(num_obs[ft][1])+"/"):
        os.makedirs(output_dir+"obs"+str(num_obs[ft][1])+"/")
    '''
    if not os.path.exists(output_dir+"obs"+str(num_obs[ft][2])+"/"):
        os.makedirs(output_dir+"obs"+str(num_obs[ft][2])+"/")
    if not os.path.exists(output_dir+"obs"+str(num_obs[ft][3])+"/"):   
        os.makedirs(output_dir+"obs"+str(num_obs[ft][3])+"/")
    
    if not os.path.exists(output_dir+"obs"+str(num_obs[ft][2])+"/TA/"):
        os.makedirs(output_dir+"obs"+str(num_obs[ft][2])+"/TA/")
    if not os.path.exists(output_dir+"obs"+str(num_obs[ft][3])+"/TA/"):   
        os.makedirs(output_dir+"obs"+str(num_obs[ft][3])+"/TA/")
    '''

In [14]:
for ft in range(nb_filter):
    
    files_uncal = sorted(glob.glob(path+'obs'+str(num_obs[ft][0])+'*/*_mirimage_uncal.fits'))
    files_bkg_uncal = sorted(glob.glob(path+'obs'+str(num_obs[ft][1])+'*/*_mirimage_uncal.fits'))
    #files_ref_uncal = sorted(glob.glob(path+'obs'+str(num_obs[ft][2])+'*/*_mirimage_uncal.fits'))
    #files_ref_bkg_uncal = sorted(glob.glob(path+'obs'+str(num_obs[ft][3])+'*/*_mirimage_uncal.fits'))
    files_TA_uncal = sorted(glob.glob(path+'obs'+str(num_obs[ft][0])+'*/TA/*02101*_mirimage_uncal.fits'))
    
    print(files_uncal)
    #To parallelise to get it more efficient ! 
    det1_dir = output_dir+'obs'+str(num_obs[ft][0])+'/'
    runmany(detector1, files_uncal)
    
    print(files_bkg_uncal)
    #To parallelise to get it more efficient ! 
    det1_dir = output_dir+'obs'+str(num_obs[ft][1])+'/'
    runmany(detector1, files_bkg_uncal)
    
    print(files_TA_uncal)
    #To parallelise to get it more efficient ! 
    det1_dir = output_dir+'obs'+str(num_obs[ft][0])+'/TA/'
    runmany(detector1, files_TA_uncal)
    
    #detector1(files_uncal, det1_dir)
    #detector1(files_bkg_uncal)
    #det1_dir = output_dir+'obs'+str(num_obs[ft][0])+'/TA/'
    #detector1(files_TA_uncal)

['/home/cperrot/Work/raw/JWST/Target/GTO/HR8799/obs16/jw01194016001_04101_00001_mirimage_uncal.fits', '/home/cperrot/Work/raw/JWST/Target/GTO/HR8799/obs16/jw01194016001_04101_00002_mirimage_uncal.fits', '/home/cperrot/Work/raw/JWST/Target/GTO/HR8799/obs16/jw01194016001_04101_00003_mirimage_uncal.fits', '/home/cperrot/Work/raw/JWST/Target/GTO/HR8799/obs16/jw01194016001_04101_00004_mirimage_uncal.fits', '/home/cperrot/Work/raw/JWST/Target/GTO/HR8799/obs16/jw01194016001_04101_00005_mirimage_uncal.fits', '/home/cperrot/Work/raw/JWST/Target/GTO/HR8799/obs16/jw01194016001_04101_00006_mirimage_uncal.fits', '/home/cperrot/Work/raw/JWST/Target/GTO/HR8799/obs16/jw01194016001_04101_00007_mirimage_uncal.fits', '/home/cperrot/Work/raw/JWST/Target/GTO/HR8799/obs16/jw01194016001_04101_00008_mirimage_uncal.fits', '/home/cperrot/Work/raw/JWST/Target/GTO/HR8799/obs16/jw01194016001_04101_00009_mirimage_uncal.fits']


2023-11-14 21:11:14,353 - stpipe.Detector1Pipeline - INFO - Detector1Pipeline instance created.
2023-11-14 21:11:14,356 - stpipe.Detector1Pipeline.group_scale - INFO - GroupScaleStep instance created.
2023-11-14 21:11:14,358 - stpipe.Detector1Pipeline.dq_init - INFO - DQInitStep instance created.
2023-11-14 21:11:14,359 - stpipe.Detector1Pipeline.saturation - INFO - SaturationStep instance created.
2023-11-14 21:11:14,361 - stpipe.Detector1Pipeline.ipc - INFO - IPCStep instance created.
2023-11-14 21:11:14,363 - stpipe.Detector1Pipeline.superbias - INFO - SuperBiasStep instance created.
2023-11-14 21:11:14,366 - stpipe.Detector1Pipeline.refpix - INFO - RefPixStep instance created.
2023-11-14 21:11:14,368 - stpipe.Detector1Pipeline.rscd - INFO - RscdStep instance created.
2023-11-14 21:11:14,371 - stpipe.Detector1Pipeline.firstframe - INFO - FirstFrameStep instance created.
2023-11-14 21:11:14,373 - stpipe.Detector1Pipeline.lastframe - INFO - LastFrameStep instance created.
2023-11-14 2

/home/cperrot/Work/raw/JWST/Target/GTO/HR8799/obs16/jw01194016001_04101_00001_mirimage_uncal.fits


2023-11-14 21:11:14,497 - stpipe.Detector1Pipeline - INFO - Step Detector1Pipeline running with args ('/home/cperrot/Work/raw/JWST/Target/GTO/HR8799/obs16/jw01194016001_04101_00001_mirimage_uncal.fits',).
2023-11-14 21:11:14,507 - stpipe.Detector1Pipeline - INFO - Step Detector1Pipeline parameters are: {'pre_hooks': [], 'post_hooks': [], 'output_file': None, 'output_dir': '/home/cperrot/Work/raw/JWST/Target/GTO/HR8799/output_with_photom/obs16/', 'output_ext': '.fits', 'output_use_model': False, 'output_use_index': True, 'save_results': True, 'skip': False, 'suffix': None, 'search_output_file': True, 'input_dir': '', 'save_calibrated_ramp': False, 'steps': {'group_scale': {'pre_hooks': [], 'post_hooks': [], 'output_file': None, 'output_dir': None, 'output_ext': '.fits', 'output_use_model': False, 'output_use_index': True, 'save_results': False, 'skip': False, 'suffix': None, 'search_output_file': True, 'input_dir': ''}, 'dq_init': {'pre_hooks': [], 'post_hooks': [], 'output_file': None,

2023-11-14 21:11:16,379 - stpipe.Detector1Pipeline - INFO - Prefetch for SATURATION reference file is '/home/cperrot/crds_cache/references/jwst/miri/jwst_miri_saturation_0034.fits'.
2023-11-14 21:11:16,380 - stpipe.Detector1Pipeline - INFO - Prefetch for SUPERBIAS reference file is 'N/A'.
2023-11-14 21:11:16,380 - stpipe.Detector1Pipeline - INFO - Prefetch for TRAPDENSITY reference file is 'N/A'.
2023-11-14 21:11:16,381 - stpipe.Detector1Pipeline - INFO - Prefetch for TRAPPARS reference file is 'N/A'.
2023-11-14 21:11:16,384 - stpipe.Detector1Pipeline - INFO - Starting calwebb_detector1 ...
2023-11-14 21:11:16,915 - stpipe.Detector1Pipeline.group_scale - INFO - Step group_scale running with args (<RampModel(2, 500, 224, 288) from jw01194016001_04101_00001_mirimage_uncal.fits>,).
2023-11-14 21:11:16,918 - stpipe.Detector1Pipeline.group_scale - INFO - Step group_scale parameters are: {'pre_hooks': [], 'post_hooks': [], 'output_file': None, 'output_dir': None, 'output_ext': '.fits', 'outp

...

/home/cperrot/Work/raw/JWST/Target/GTO/HR8799/obs16/TA/jw01194016001_02101_00002_mirimage_uncal.fits


2023-11-14 23:03:54,894 - stpipe.Detector1Pipeline - INFO - Step Detector1Pipeline running with args ('/home/cperrot/Work/raw/JWST/Target/GTO/HR8799/obs16/TA/jw01194016001_02101_00002_mirimage_uncal.fits',).
2023-11-14 23:03:54,905 - stpipe.Detector1Pipeline - INFO - Step Detector1Pipeline parameters are: {'pre_hooks': [], 'post_hooks': [], 'output_file': None, 'output_dir': '/home/cperrot/Work/raw/JWST/Target/GTO/HR8799/output_with_photom/obs16/TA/', 'output_ext': '.fits', 'output_use_model': False, 'output_use_index': True, 'save_results': True, 'skip': False, 'suffix': None, 'search_output_file': True, 'input_dir': '', 'save_calibrated_ramp': False, 'steps': {'group_scale': {'pre_hooks': [], 'post_hooks': [], 'output_file': None, 'output_dir': None, 'output_ext': '.fits', 'output_use_model': False, 'output_use_index': True, 'save_results': False, 'skip': False, 'suffix': None, 'search_output_file': True, 'input_dir': ''}, 'dq_init': {'pre_hooks': [], 'post_hooks': [], 'output_file':

2023-11-14 23:03:55,149 - stpipe.Detector1Pipeline - INFO - Prefetch for SATURATION reference file is '/home/cperrot/crds_cache/references/jwst/miri/jwst_miri_saturation_0034.fits'.
2023-11-14 23:03:55,150 - stpipe.Detector1Pipeline - INFO - Prefetch for SUPERBIAS reference file is 'N/A'.
2023-11-14 23:03:55,150 - stpipe.Detector1Pipeline - INFO - Prefetch for TRAPDENSITY reference file is 'N/A'.
2023-11-14 23:03:55,151 - stpipe.Detector1Pipeline - INFO - Prefetch for TRAPPARS reference file is 'N/A'.
2023-11-14 23:03:55,152 - stpipe.Detector1Pipeline - INFO - Starting calwebb_detector1 ...
2023-11-14 23:03:55,440 - stpipe.Detector1Pipeline.group_scale - INFO - Step group_scale running with args (<RampModel(1, 44, 224, 288) from jw01194016001_02101_00002_mirimage_uncal.fits>,).
2023-11-14 23:03:55,442 - stpipe.Detector1Pipeline.group_scale - INFO - Step group_scale parameters are: {'pre_hooks': [], 'post_hooks': [], 'output_file': None, 'output_dir': None, 'output_ext': '.fits', 'outpu

2023-11-14 23:03:56,737 - stpipe.Detector1Pipeline.linearity - WARNING - Keyword RESERVED_4 does not correspond to an existing DQ mnemonic, so will be ignored
2023-11-14 23:03:56,752 - stpipe.Detector1Pipeline.linearity - WARNING - Keyword UNRELIABLE_RESET does not correspond to an existing DQ mnemonic, so will be ignored
2023-11-14 23:03:56,833 - stpipe.Detector1Pipeline.linearity - INFO - Step linearity done
2023-11-14 23:03:56,901 - stpipe.Detector1Pipeline.rscd - INFO - Step rscd running with args (<RampModel(1, 44, 224, 288) from jw01194016001_02101_00002_mirimage_uncal.fits>,).
2023-11-14 23:03:56,904 - stpipe.Detector1Pipeline.rscd - INFO - Step rscd parameters are: {'pre_hooks': [], 'post_hooks': [], 'output_file': None, 'output_dir': None, 'output_ext': '.fits', 'output_use_model': False, 'output_use_index': True, 'save_results': False, 'skip': False, 'suffix': None, 'search_output_file': True, 'input_dir': '/home/cperrot/Work/raw/JWST/Target/GTO/HR8799/obs16/TA', 'type': 'bas

2023-11-14 23:03:58,520 - stpipe.Detector1Pipeline.ramp_fit - INFO - Extracting gain subarray to match science data
2023-11-14 23:03:58,522 - stpipe.Detector1Pipeline.ramp_fit - INFO - Extracting readnoise subarray to match science data
2023-11-14 23:03:58,523 - stpipe.Detector1Pipeline.ramp_fit - INFO - Using algorithm = ols
2023-11-14 23:03:58,524 - stpipe.Detector1Pipeline.ramp_fit - INFO - Using weighting = optimal
2023-11-14 23:03:58,581 - stpipe.Detector1Pipeline.ramp_fit - INFO - Number of leading groups that are flagged as DO_NOT_USE: 1
2023-11-14 23:03:58,582 - stpipe.Detector1Pipeline.ramp_fit - INFO - MIRI dataset has all pixels in the final group flagged as DO_NOT_USE.
2023-11-14 23:04:01,102 - stpipe.Detector1Pipeline.ramp_fit - INFO - Number of groups per integration: 42
2023-11-14 23:04:01,103 - stpipe.Detector1Pipeline.ramp_fit - INFO - Number of integrations: 1
2023-11-14 23:04:01,190 - stpipe.Detector1Pipeline.ramp_fit - INFO - Step ramp_fit done
2023-11-14 23:04:01,2

In [15]:
print("c'est fini")

c'est fini


### STAGE 2

In [6]:
#Define path
path = "/home/cperrot/Work/raw/JWST/Target/GTO/GJ504/output_with_photom/"
output_dir = path
name_pl = 'GJ504'
filter_name = ["F1065C","F1140C","F1550C"] #["F1065C","F1140C","F1550C"]#, "F2100W"]
cmaps = ["Blues_r","Greens_r","Oranges_r"]

#Number of the observations : useful to retrieve the data 
#For each filter : science obs, background, ref, and ref background
num_obs = [[13,18],[14,17],[15,16]] 

D = 6.57 #Diameter of the telescope
wavelength_filter = [10.65,11.40,15.50]
nb_filter = len(wavelength_filter)
fwhm = [((wavelength_filter[t]*10**-6)/6.57)*(180*3600/np.pi) for t in range(nb_filter)]
pix_size = 0.11 #arcsec/pixel

#Center corono : DEC, RA 
yxc1065 = [112.00, 120.25]  
yxc1140 = [112.16, 119.77]  
yxc1550 = [113.20, 119.73] 
yxc2330 = [167.88, 144.33]  
center_corono = (yxc1065,yxc1140,yxc1550)

In [11]:
for ft in range(len(num_obs)):

    #Retrieve from MAST : 
    files_rate = sorted(glob.glob(path+'obs'+str(num_obs[ft][0])+'*/*_mirimage_rate.fits'))
    files_bkg_rate = sorted(glob.glob(path+'obs'+str(num_obs[ft][1])+'*/*_mirimage_rate.fits'))
    #files_ref_rate = sorted(glob.glob(path+'obs'+str(num_obs[ft][2])+'*/*_mirimage_rate.fits'))
    #files_ref_bkg_rate = sorted(glob.glob(path+'obs'+str(num_obs[ft][3])+'*/*_mirimage_rate.fits'))

    files_TA = sorted(glob.glob(path+'obs'+str(num_obs[ft][0])+'*/TA/*02101*_mirimage_rate.fits'))
       
    #Liste ASN with science and bkg files : 
    asnlist = writeasnfile(files_rate, files_bkg_rate, name_pl)
    #Run the pipeline : 
    image2(asnlist, output_dir+"obs"+str(num_obs[ft][0])+"/")

    #Same for ref files
    #asnlist_ref = writeasnfile(files_ref_rate, files_ref_bkg_rate,name_pl)
    #image2(asnlist_ref,output_dir+"obs"+str(num_obs[ft][2])+"/")
    
    #Same for TA (useful for photometry only)
    asnlist2 = writeasnfile2(files_TA,name_pl)
    image2(asnlist2, output_dir+"obs"+str(num_obs[ft][0])+"/TA/")

2023-11-16 15:40:08,057 - stpipe - WARNING - /home/cperrot/anaconda3/envs/mirisim/lib/python3.9/site-packages/jwst/associations/association.py:215: UserWarning: 'expname' contains path, but should only be a filename.  All input files should be in a single directory, so no path is needed.
  warnings.warn(err_str, UserWarning)

2023-11-16 15:40:08,063 - stpipe.Image2Pipeline - INFO - Image2Pipeline instance created.
2023-11-16 15:40:08,065 - stpipe.Image2Pipeline.bkg_subtract - INFO - BackgroundStep instance created.
2023-11-16 15:40:08,068 - stpipe.Image2Pipeline.assign_wcs - INFO - AssignWcsStep instance created.
2023-11-16 15:40:08,069 - stpipe.Image2Pipeline.flat_field - INFO - FlatFieldStep instance created.
2023-11-16 15:40:08,072 - stpipe.Image2Pipeline.photom - INFO - PhotomStep instance created.
2023-11-16 15:40:08,074 - stpipe.Image2Pipeline.resample - INFO - ResampleStep instance created.
2023-11-16 15:40:08,166 - stpipe.Image2Pipeline - INFO - Step Image2Pipeline running with

{
    "asn_type": "image2",
    "asn_rule": "DMSLevel2bBase",
    "version_id": null,
    "code_version": "1.12.5",
    "degraded_status": "No known degraded exposures in association.",
    "program": "noprogram",
    "constraints": "No constraints",
    "asn_id": "a3001",
    "asn_pool": "GJ504_pool",
    "products": [
        {
            "name": "GJ504_dither1",
            "members": [
                {
                    "expname": "/home/cperrot/Work/raw/JWST/Target/GTO/GJ504/output_with_photom/obs13/jw01277013001_04101_00001_mirimage_rate.fits",
                    "exptype": "science"
                },
                {
                    "expname": "/home/cperrot/Work/raw/JWST/Target/GTO/GJ504/output_with_photom/obs18/jw01277018001_02101_00001_mirimage_rate.fits",
                    "exptype": "background"
                },
                {
                    "expname": "/home/cperrot/Work/raw/JWST/Target/GTO/GJ504/output_with_photom/obs18/jw01277018001_02101_00002_mir

2023-11-16 15:40:08,320 - stpipe.Image2Pipeline.bkg_subtract - INFO - Step bkg_subtract running with args (<ImageModel(224, 288) from jw01277013001_04101_00001_mirimage_rate.fits>, ['/home/cperrot/Work/raw/JWST/Target/GTO/GJ504/output_with_photom/obs18/jw01277018001_02101_00001_mirimage_rate.fits', '/home/cperrot/Work/raw/JWST/Target/GTO/GJ504/output_with_photom/obs18/jw01277018001_02101_00002_mirimage_rate.fits']).
2023-11-16 15:40:08,322 - stpipe.Image2Pipeline.bkg_subtract - INFO - Step bkg_subtract parameters are: {'pre_hooks': [], 'post_hooks': [], 'output_file': None, 'output_dir': None, 'output_ext': '.fits', 'output_use_model': False, 'output_use_index': True, 'save_results': False, 'skip': False, 'suffix': 'bsub', 'search_output_file': True, 'input_dir': '', 'save_combined_background': False, 'sigma': 3.0, 'maxiters': None, 'wfss_mmag_extract': None}
2023-11-16 15:40:08,350 - stpipe.Image2Pipeline.bkg_subtract - INFO - Accumulate bkg from /home/cperrot/Work/raw/JWST/Target/GTO

2023-11-16 15:40:09,640 - stpipe.Image2Pipeline - INFO - Prefetching reference files for dataset: 'jw01277013001_02101_00001_mirimage_rate.fits' reftypes = ['area', 'camera', 'collimator', 'disperser', 'distortion', 'drizpars', 'filteroffset', 'fore', 'fpa', 'ifufore', 'ifupost', 'ifuslicer', 'msa', 'ote', 'photom', 'regions', 'specwcs', 'wavelengthrange', 'wfssbkg']
2023-11-16 15:40:09,644 - stpipe.Image2Pipeline - INFO - Prefetch for AREA reference file is 'N/A'.
2023-11-16 15:40:09,644 - stpipe.Image2Pipeline - INFO - Prefetch for CAMERA reference file is 'N/A'.
2023-11-16 15:40:09,645 - stpipe.Image2Pipeline - INFO - Prefetch for COLLIMATOR reference file is 'N/A'.
2023-11-16 15:40:09,645 - stpipe.Image2Pipeline - INFO - Prefetch for DISPERSER reference file is 'N/A'.
2023-11-16 15:40:09,646 - stpipe.Image2Pipeline - INFO - Prefetch for DISTORTION reference file is '/home/cperrot/crds_cache/references/jwst/miri/jwst_miri_distortion_0047.asdf'.
2023-11-16 15:40:09,646 - stpipe.Image

{
    "asn_type": "image2",
    "asn_rule": "DMSLevel2bBase",
    "version_id": null,
    "code_version": "1.12.5",
    "degraded_status": "No known degraded exposures in association.",
    "program": "noprogram",
    "constraints": "No constraints",
    "asn_id": "a3001",
    "asn_pool": "GJ504_pool",
    "products": [
        {
            "name": "GJ504_dither1",
            "members": [
                {
                    "expname": "/home/cperrot/Work/raw/JWST/Target/GTO/GJ504/output_with_photom/obs13/TA/jw01277013001_02101_00001_mirimage_rate.fits",
                    "exptype": "science"
                }
            ]
        },
        {
            "name": "GJ504_dither2",
            "members": [
                {
                    "expname": "/home/cperrot/Work/raw/JWST/Target/GTO/GJ504/output_with_photom/obs13/TA/jw01277013001_02101_00002_mirimage_rate.fits",
                    "exptype": "science"
                }
            ]
        }
    ]
}


...

{
    "asn_type": "image2",
    "asn_rule": "DMSLevel2bBase",
    "version_id": null,
    "code_version": "1.12.5",
    "degraded_status": "No known degraded exposures in association.",
    "program": "noprogram",
    "constraints": "No constraints",
    "asn_id": "a3001",
    "asn_pool": "GJ504_pool",
    "products": [
        {
            "name": "GJ504_dither1",
            "members": [
                {
                    "expname": "/home/cperrot/Work/raw/JWST/Target/GTO/GJ504/output_with_photom/obs15/jw01277015001_04101_00001_mirimage_rate.fits",
                    "exptype": "science"
                },
                {
                    "expname": "/home/cperrot/Work/raw/JWST/Target/GTO/GJ504/output_with_photom/obs16/jw01277016001_02101_00001_mirimage_rate.fits",
                    "exptype": "background"
                },
                {
                    "expname": "/home/cperrot/Work/raw/JWST/Target/GTO/GJ504/output_with_photom/obs16/jw01277016001_02101_00002_mir

2023-11-16 15:40:16,705 - stpipe.Image2Pipeline.bkg_subtract - INFO - Step bkg_subtract running with args (<ImageModel(224, 288) from jw01277015001_04101_00001_mirimage_rate.fits>, ['/home/cperrot/Work/raw/JWST/Target/GTO/GJ504/output_with_photom/obs16/jw01277016001_02101_00001_mirimage_rate.fits', '/home/cperrot/Work/raw/JWST/Target/GTO/GJ504/output_with_photom/obs16/jw01277016001_02101_00002_mirimage_rate.fits']).
2023-11-16 15:40:16,706 - stpipe.Image2Pipeline.bkg_subtract - INFO - Step bkg_subtract parameters are: {'pre_hooks': [], 'post_hooks': [], 'output_file': None, 'output_dir': None, 'output_ext': '.fits', 'output_use_model': False, 'output_use_index': True, 'save_results': False, 'skip': False, 'suffix': 'bsub', 'search_output_file': True, 'input_dir': '', 'save_combined_background': False, 'sigma': 3.0, 'maxiters': None, 'wfss_mmag_extract': None}
2023-11-16 15:40:16,739 - stpipe.Image2Pipeline.bkg_subtract - INFO - Accumulate bkg from /home/cperrot/Work/raw/JWST/Target/GTO

{
    "asn_type": "image2",
    "asn_rule": "DMSLevel2bBase",
    "version_id": null,
    "code_version": "1.12.5",
    "degraded_status": "No known degraded exposures in association.",
    "program": "noprogram",
    "constraints": "No constraints",
    "asn_id": "a3001",
    "asn_pool": "GJ504_pool",
    "products": [
        {
            "name": "GJ504_dither1",
            "members": [
                {
                    "expname": "/home/cperrot/Work/raw/JWST/Target/GTO/GJ504/output_with_photom/obs15/TA/jw01277015001_02101_00001_mirimage_rate.fits",
                    "exptype": "science"
                }
            ]
        },
        {
            "name": "GJ504_dither2",
            "members": [
                {
                    "expname": "/home/cperrot/Work/raw/JWST/Target/GTO/GJ504/output_with_photom/obs15/TA/jw01277015001_02101_00002_mirimage_rate.fits",
                    "exptype": "science"
                }
            ]
        }
    ]
}


2023-11-16 15:40:18,486 - stpipe.Image2Pipeline - INFO - Prefetching reference files for dataset: 'jw01277015001_02101_00001_mirimage_rate.fits' reftypes = ['area', 'camera', 'collimator', 'disperser', 'distortion', 'drizpars', 'filteroffset', 'fore', 'fpa', 'ifufore', 'ifupost', 'ifuslicer', 'msa', 'ote', 'photom', 'regions', 'specwcs', 'wavelengthrange', 'wfssbkg']
2023-11-16 15:40:18,491 - stpipe.Image2Pipeline - INFO - Prefetch for AREA reference file is 'N/A'.
2023-11-16 15:40:18,491 - stpipe.Image2Pipeline - INFO - Prefetch for CAMERA reference file is 'N/A'.
2023-11-16 15:40:18,492 - stpipe.Image2Pipeline - INFO - Prefetch for COLLIMATOR reference file is 'N/A'.
2023-11-16 15:40:18,492 - stpipe.Image2Pipeline - INFO - Prefetch for DISPERSER reference file is 'N/A'.
2023-11-16 15:40:18,492 - stpipe.Image2Pipeline - INFO - Prefetch for DISTORTION reference file is '/home/cperrot/crds_cache/references/jwst/miri/jwst_miri_distortion_0047.asdf'.
2023-11-16 15:40:18,493 - stpipe.Image

2023-11-16 15:40:19,845 - stpipe.Image2Pipeline.assign_wcs - INFO - Created a MIRI mir_tacq pipeline with references {'distortion': '/home/cperrot/crds_cache/references/jwst/miri/jwst_miri_distortion_0047.asdf', 'filteroffset': '/home/cperrot/crds_cache/references/jwst/miri/jwst_miri_filteroffset_0007.asdf', 'specwcs': None, 'regions': None, 'wavelengthrange': None, 'camera': None, 'collimator': None, 'disperser': None, 'fore': None, 'fpa': None, 'msa': None, 'ote': None, 'ifupost': None, 'ifufore': None, 'ifuslicer': None}
2023-11-16 15:40:19,918 - stpipe.Image2Pipeline.assign_wcs - INFO - Update S_REGION to POLYGON ICRS  199.186271963 9.423706574 199.192427022 9.420450392 199.196715313 9.428165031 199.190547153 9.431448578
2023-11-16 15:40:19,919 - stpipe.Image2Pipeline.assign_wcs - INFO - assign_wcs updated S_REGION to POLYGON ICRS  199.186271963 9.423706574 199.192427022 9.420450392 199.196715313 9.428165031 199.190547153 9.431448578
2023-11-16 15:40:19,919 - stpipe.Image2Pipeline.

In [14]:
#Define path
path = "/home/cperrot/Work/raw/JWST/Target/GTO/HD95086/output_with_photom/"
output_dir = path
name_pl = 'HD310459'
filter_name = ["F1065C","F1140C"] #["F1065C","F1140C","F1550C"]#, "F2100W"]
cmaps = ["Blues_r","Greens_r","Oranges_r"]

#Number of the observations : useful to retrieve the data 
#For each filter : science obs, background, ref, and ref background
num_obs = [[7,12],[8,11]] 

D = 6.57 #Diameter of the telescope
wavelength_filter = [10.65,11.40]
nb_filter = len(wavelength_filter)
fwhm = [((wavelength_filter[t]*10**-6)/6.57)*(180*3600/np.pi) for t in range(nb_filter)]
pix_size = 0.11 #arcsec/pixel

#Center corono : DEC, RA 
yxc1065 = [112.00, 120.25]  
yxc1140 = [112.16, 119.77]  
yxc1550 = [113.20, 119.73] 
yxc2330 = [167.88, 144.33]  
center_corono = (yxc1065,yxc1140,yxc1550)

In [15]:
for ft in range(len(num_obs)):

    #Retrieve from MAST : 
    files_rate = sorted(glob.glob(path+'obs'+str(num_obs[ft][0])+'*/*_mirimage_rate.fits'))
    files_bkg_rate = sorted(glob.glob(path+'obs'+str(num_obs[ft][1])+'*/*_mirimage_rate.fits'))
    #files_ref_rate = sorted(glob.glob(path+'obs'+str(num_obs[ft][2])+'*/*_mirimage_rate.fits'))
    #files_ref_bkg_rate = sorted(glob.glob(path+'obs'+str(num_obs[ft][3])+'*/*_mirimage_rate.fits'))

    files_TA = sorted(glob.glob(path+'obs'+str(num_obs[ft][0])+'*/TA/*02101*_mirimage_rate.fits'))
       
    #Liste ASN with science and bkg files : 
    asnlist = writeasnfile(files_rate, files_bkg_rate, name_pl)
    #Run the pipeline : 
    image2(asnlist, output_dir+"obs"+str(num_obs[ft][0])+"/")

    #Same for ref files
    #asnlist_ref = writeasnfile(files_ref_rate, files_ref_bkg_rate,name_pl)
    #image2(asnlist_ref,output_dir+"obs"+str(num_obs[ft][2])+"/")
    
    #Same for TA (useful for photometry only)
    asnlist2 = writeasnfile2(files_TA,name_pl)
    image2(asnlist2, output_dir+"obs"+str(num_obs[ft][0])+"/TA/")

2023-11-16 15:53:39,147 - stpipe - WARNING - /home/cperrot/anaconda3/envs/mirisim/lib/python3.9/site-packages/jwst/associations/association.py:215: UserWarning: 'expname' contains path, but should only be a filename.  All input files should be in a single directory, so no path is needed.
  warnings.warn(err_str, UserWarning)

2023-11-16 15:53:39,153 - stpipe.Image2Pipeline - INFO - Image2Pipeline instance created.
2023-11-16 15:53:39,154 - stpipe.Image2Pipeline.bkg_subtract - INFO - BackgroundStep instance created.
2023-11-16 15:53:39,156 - stpipe.Image2Pipeline.assign_wcs - INFO - AssignWcsStep instance created.
2023-11-16 15:53:39,157 - stpipe.Image2Pipeline.flat_field - INFO - FlatFieldStep instance created.
2023-11-16 15:53:39,159 - stpipe.Image2Pipeline.photom - INFO - PhotomStep instance created.
2023-11-16 15:53:39,160 - stpipe.Image2Pipeline.resample - INFO - ResampleStep instance created.
2023-11-16 15:53:39,287 - stpipe.Image2Pipeline - INFO - Step Image2Pipeline running with

{
    "asn_type": "image2",
    "asn_rule": "DMSLevel2bBase",
    "version_id": null,
    "code_version": "1.12.5",
    "degraded_status": "No known degraded exposures in association.",
    "program": "noprogram",
    "constraints": "No constraints",
    "asn_id": "a3001",
    "asn_pool": "HD310459_pool",
    "products": [
        {
            "name": "HD310459_dither1",
            "members": [
                {
                    "expname": "/home/cperrot/Work/raw/JWST/Target/GTO/HD95086/output_with_photom/obs7/jw01277007001_04101_00001_mirimage_rate.fits",
                    "exptype": "science"
                },
                {
                    "expname": "/home/cperrot/Work/raw/JWST/Target/GTO/HD95086/output_with_photom/obs12/jw01277012001_02101_00001_mirimage_rate.fits",
                    "exptype": "background"
                },
                {
                    "expname": "/home/cperrot/Work/raw/JWST/Target/GTO/HD95086/output_with_photom/obs12/jw01277012001_0210

2023-11-16 15:53:39,485 - stpipe.Image2Pipeline.bkg_subtract - INFO - Step bkg_subtract running with args (<ImageModel(224, 288) from jw01277007001_04101_00001_mirimage_rate.fits>, ['/home/cperrot/Work/raw/JWST/Target/GTO/HD95086/output_with_photom/obs12/jw01277012001_02101_00001_mirimage_rate.fits', '/home/cperrot/Work/raw/JWST/Target/GTO/HD95086/output_with_photom/obs12/jw01277012001_02101_00002_mirimage_rate.fits']).
2023-11-16 15:53:39,487 - stpipe.Image2Pipeline.bkg_subtract - INFO - Step bkg_subtract parameters are: {'pre_hooks': [], 'post_hooks': [], 'output_file': None, 'output_dir': None, 'output_ext': '.fits', 'output_use_model': False, 'output_use_index': True, 'save_results': False, 'skip': False, 'suffix': 'bsub', 'search_output_file': True, 'input_dir': '', 'save_combined_background': False, 'sigma': 3.0, 'maxiters': None, 'wfss_mmag_extract': None}
2023-11-16 15:53:39,515 - stpipe.Image2Pipeline.bkg_subtract - INFO - Accumulate bkg from /home/cperrot/Work/raw/JWST/Target

2023-11-16 15:53:41,383 - stpipe.Image2Pipeline.assign_wcs - INFO - Created a MIRI mir_4qpm pipeline with references {'distortion': '/home/cperrot/crds_cache/references/jwst/miri/jwst_miri_distortion_0047.asdf', 'filteroffset': '/home/cperrot/crds_cache/references/jwst/miri/jwst_miri_filteroffset_0009.asdf', 'specwcs': None, 'regions': None, 'wavelengthrange': None, 'camera': None, 'collimator': None, 'disperser': None, 'fore': None, 'fpa': None, 'msa': None, 'ote': None, 'ifupost': None, 'ifufore': None, 'ifuslicer': None}
2023-11-16 15:53:41,467 - stpipe.Image2Pipeline.assign_wcs - INFO - Update S_REGION to POLYGON ICRS  168.547151332 -68.574057959 168.551049557 -68.567447841 168.527417712 -68.565635059 168.523520989 -68.572290231
2023-11-16 15:53:41,467 - stpipe.Image2Pipeline.assign_wcs - INFO - assign_wcs updated S_REGION to POLYGON ICRS  168.547151332 -68.574057959 168.551049557 -68.567447841 168.527417712 -68.565635059 168.523520989 -68.572290231
2023-11-16 15:53:41,468 - stpipe

2023-11-16 15:53:43,042 - stpipe.Image2Pipeline.photom - INFO - Step photom parameters are: {'pre_hooks': [], 'post_hooks': [], 'output_file': None, 'output_dir': None, 'output_ext': '.fits', 'output_use_model': False, 'output_use_index': True, 'save_results': False, 'skip': False, 'suffix': 'photom', 'search_output_file': True, 'input_dir': '', 'inverse': False, 'source_type': None, 'mrs_time_correction': True}
2023-11-16 15:53:43,074 - stpipe.Image2Pipeline.photom - INFO - Using photom reference file: /home/cperrot/crds_cache/references/jwst/miri/jwst_miri_photom_0186.fits
2023-11-16 15:53:43,074 - stpipe.Image2Pipeline.photom - INFO - Using area reference file: N/A
2023-11-16 15:53:43,165 - stpipe.Image2Pipeline.photom - INFO - Using instrument: MIRI
2023-11-16 15:53:43,166 - stpipe.Image2Pipeline.photom - INFO -  detector: MIRIMAGE
2023-11-16 15:53:43,166 - stpipe.Image2Pipeline.photom - INFO -  exp_type: MIR_4QPM
2023-11-16 15:53:43,167 - stpipe.Image2Pipeline.photom - INFO -  fil

...

2023-11-16 15:54:03,748 - stpipe.Image2Pipeline.bkg_subtract - INFO - Accumulate bkg from /home/cperrot/Work/raw/JWST/Target/GTO/HD95086/output_with_photom/obs11/jw01277011001_02101_00002_mirimage_rate.fits
2023-11-16 15:54:03,810 - stpipe.Image2Pipeline.bkg_subtract - WARNING - /home/cperrot/anaconda3/envs/mirisim/lib/python3.9/site-packages/astropy/stats/sigma_clipping.py:364: AstropyUserWarning: Input data contains invalid values (NaNs or infs), which were automatically clipped.
  warnings.warn(

2023-11-16 15:54:03,868 - stpipe.Image2Pipeline.bkg_subtract - INFO - Step bkg_subtract done
2023-11-16 15:54:03,994 - stpipe.Image2Pipeline.assign_wcs - INFO - Step assign_wcs running with args (<ImageModel(224, 288) from jw01277008001_04101_00009_mirimage_rate.fits>,).
2023-11-16 15:54:03,996 - stpipe.Image2Pipeline.assign_wcs - INFO - Step assign_wcs parameters are: {'pre_hooks': [], 'post_hooks': [], 'output_file': None, 'output_dir': None, 'output_ext': '.fits', 'output_use_model': Fal

2023-11-16 15:54:06,830 - stpipe.Image2Pipeline - INFO - Step Image2Pipeline running with args ('HD310459_lvl2_asn.json',).
2023-11-16 15:54:06,836 - stpipe.Image2Pipeline - INFO - Step Image2Pipeline parameters are: {'pre_hooks': [], 'post_hooks': [], 'output_file': None, 'output_dir': '/home/cperrot/Work/raw/JWST/Target/GTO/HD95086/output_with_photom/obs8/TA/', 'output_ext': '.fits', 'output_use_model': False, 'output_use_index': True, 'save_results': True, 'skip': False, 'suffix': None, 'search_output_file': True, 'input_dir': '', 'save_bsub': False, 'steps': {'bkg_subtract': {'pre_hooks': [], 'post_hooks': [], 'output_file': None, 'output_dir': None, 'output_ext': '.fits', 'output_use_model': False, 'output_use_index': True, 'save_results': False, 'skip': False, 'suffix': None, 'search_output_file': True, 'input_dir': '', 'save_combined_background': False, 'sigma': 3.0, 'maxiters': None, 'wfss_mmag_extract': None}, 'assign_wcs': {'pre_hooks': [], 'post_hooks': [], 'output_file': No

{
    "asn_type": "image2",
    "asn_rule": "DMSLevel2bBase",
    "version_id": null,
    "code_version": "1.12.5",
    "degraded_status": "No known degraded exposures in association.",
    "program": "noprogram",
    "constraints": "No constraints",
    "asn_id": "a3001",
    "asn_pool": "HD310459_pool",
    "products": [
        {
            "name": "HD310459_dither1",
            "members": [
                {
                    "expname": "/home/cperrot/Work/raw/JWST/Target/GTO/HD95086/output_with_photom/obs8/TA/jw01277008001_02101_00001_mirimage_rate.fits",
                    "exptype": "science"
                }
            ]
        },
        {
            "name": "HD310459_dither2",
            "members": [
                {
                    "expname": "/home/cperrot/Work/raw/JWST/Target/GTO/HD95086/output_with_photom/obs8/TA/jw01277008001_02101_00002_mirimage_rate.fits",
                    "exptype": "science"
                }
            ]
        }
    ]
}


2023-11-16 15:54:06,943 - stpipe.Image2Pipeline - INFO - Prefetching reference files for dataset: 'jw01277008001_02101_00001_mirimage_rate.fits' reftypes = ['area', 'camera', 'collimator', 'disperser', 'distortion', 'drizpars', 'filteroffset', 'fore', 'fpa', 'ifufore', 'ifupost', 'ifuslicer', 'msa', 'ote', 'photom', 'regions', 'specwcs', 'wavelengthrange', 'wfssbkg']
2023-11-16 15:54:06,946 - stpipe.Image2Pipeline - INFO - Prefetch for AREA reference file is 'N/A'.
2023-11-16 15:54:06,947 - stpipe.Image2Pipeline - INFO - Prefetch for CAMERA reference file is 'N/A'.
2023-11-16 15:54:06,948 - stpipe.Image2Pipeline - INFO - Prefetch for COLLIMATOR reference file is 'N/A'.
2023-11-16 15:54:06,948 - stpipe.Image2Pipeline - INFO - Prefetch for DISPERSER reference file is 'N/A'.
2023-11-16 15:54:06,948 - stpipe.Image2Pipeline - INFO - Prefetch for DISTORTION reference file is '/home/cperrot/crds_cache/references/jwst/miri/jwst_miri_distortion_0047.asdf'.
2023-11-16 15:54:06,949 - stpipe.Image

2023-11-16 15:54:08,245 - stpipe.Image2Pipeline.assign_wcs - INFO - Created a MIRI mir_tacq pipeline with references {'distortion': '/home/cperrot/crds_cache/references/jwst/miri/jwst_miri_distortion_0047.asdf', 'filteroffset': '/home/cperrot/crds_cache/references/jwst/miri/jwst_miri_filteroffset_0010.asdf', 'specwcs': None, 'regions': None, 'wavelengthrange': None, 'camera': None, 'collimator': None, 'disperser': None, 'fore': None, 'fpa': None, 'msa': None, 'ote': None, 'ifupost': None, 'ifufore': None, 'ifuslicer': None}
2023-11-16 15:54:08,310 - stpipe.Image2Pipeline.assign_wcs - INFO - Update S_REGION to POLYGON ICRS  168.548290005 -68.574707363 168.552173284 -68.567990639 168.528584027 -68.566136078 168.524656254 -68.572884943
2023-11-16 15:54:08,311 - stpipe.Image2Pipeline.assign_wcs - INFO - assign_wcs updated S_REGION to POLYGON ICRS  168.548290005 -68.574707363 168.552173284 -68.567990639 168.528584027 -68.566136078 168.524656254 -68.572884943
2023-11-16 15:54:08,311 - stpipe

In [3]:
#Define path
path = "/home/cperrot/Work/raw/JWST/Target/GTO/HR8799/output_with_photom/"
output_dir = path
name_pl = 'HD218261'
filter_name = ["F1550C"] #["F1065C","F1140C","F1550C"]#, "F2100W"]
cmaps = ["Blues_r","Greens_r","Oranges_r"]

#Number of the observations : useful to retrieve the data 
#For each filter : science obs, background, ref, and ref background
num_obs = [[16,18]] 

D = 6.57 #Diameter of the telescope
wavelength_filter = [15.50]
nb_filter = len(wavelength_filter)
fwhm = [((wavelength_filter[t]*10**-6)/6.57)*(180*3600/np.pi) for t in range(nb_filter)]
pix_size = 0.11 #arcsec/pixel

#Center corono : DEC, RA 
yxc1065 = [112.00, 120.25]  
yxc1140 = [112.16, 119.77]  
yxc1550 = [113.20, 119.73] 
yxc2330 = [167.88, 144.33]  
center_corono = (yxc1065,yxc1140,yxc1550)

In [4]:
for ft in range(len(num_obs)):

    #Retrieve from MAST : 
    files_rate = sorted(glob.glob(path+'obs'+str(num_obs[ft][0])+'*/*_mirimage_rate.fits'))
    files_bkg_rate = sorted(glob.glob(path+'obs'+str(num_obs[ft][1])+'*/*_mirimage_rate.fits'))
    #files_ref_rate = sorted(glob.glob(path+'obs'+str(num_obs[ft][2])+'*/*_mirimage_rate.fits'))
    #files_ref_bkg_rate = sorted(glob.glob(path+'obs'+str(num_obs[ft][3])+'*/*_mirimage_rate.fits'))

    files_TA = sorted(glob.glob(path+'obs'+str(num_obs[ft][0])+'*/TA/*02101*_mirimage_rate.fits'))
       
    #Liste ASN with science and bkg files : 
    asnlist = writeasnfile(files_rate, files_bkg_rate, name_pl)
    #Run the pipeline : 
    image2(asnlist, output_dir+"obs"+str(num_obs[ft][0])+"/")

    #Same for ref files
    #asnlist_ref = writeasnfile(files_ref_rate, files_ref_bkg_rate,name_pl)
    #image2(asnlist_ref,output_dir+"obs"+str(num_obs[ft][2])+"/")
    
    #Same for TA (useful for photometry only)
    asnlist2 = writeasnfile2(files_TA,name_pl)
    image2(asnlist2, output_dir+"obs"+str(num_obs[ft][0])+"/TA/")

2023-11-16 15:57:50,566 - stpipe - WARNING - /home/cperrot/anaconda3/envs/mirisim/lib/python3.9/site-packages/jwst/associations/association.py:215: UserWarning: 'expname' contains path, but should only be a filename.  All input files should be in a single directory, so no path is needed.
  warnings.warn(err_str, UserWarning)

2023-11-16 15:57:50,574 - stpipe.Image2Pipeline - INFO - Image2Pipeline instance created.
2023-11-16 15:57:50,575 - stpipe.Image2Pipeline.bkg_subtract - INFO - BackgroundStep instance created.
2023-11-16 15:57:50,577 - stpipe.Image2Pipeline.assign_wcs - INFO - AssignWcsStep instance created.
2023-11-16 15:57:50,578 - stpipe.Image2Pipeline.flat_field - INFO - FlatFieldStep instance created.
2023-11-16 15:57:50,580 - stpipe.Image2Pipeline.photom - INFO - PhotomStep instance created.
2023-11-16 15:57:50,581 - stpipe.Image2Pipeline.resample - INFO - ResampleStep instance created.
2023-11-16 15:57:50,653 - stpipe.Image2Pipeline - INFO - Step Image2Pipeline running with

{
    "asn_type": "image2",
    "asn_rule": "DMSLevel2bBase",
    "version_id": null,
    "code_version": "1.12.5",
    "degraded_status": "No known degraded exposures in association.",
    "program": "noprogram",
    "constraints": "No constraints",
    "asn_id": "a3001",
    "asn_pool": "HD218261_pool",
    "products": [
        {
            "name": "HD218261_dither1",
            "members": [
                {
                    "expname": "/home/cperrot/Work/raw/JWST/Target/GTO/HR8799/output_with_photom/obs16/jw01194016001_04101_00001_mirimage_rate.fits",
                    "exptype": "science"
                },
                {
                    "expname": "/home/cperrot/Work/raw/JWST/Target/GTO/HR8799/output_with_photom/obs18/jw01194018001_02101_00001_mirimage_rate.fits",
                    "exptype": "background"
                },
                {
                    "expname": "/home/cperrot/Work/raw/JWST/Target/GTO/HR8799/output_with_photom/obs18/jw01194018001_02101_

2023-11-16 15:57:51,433 - stpipe.Image2Pipeline.bkg_subtract - INFO - Step bkg_subtract running with args (<ImageModel(224, 288) from jw01194016001_04101_00001_mirimage_rate.fits>, ['/home/cperrot/Work/raw/JWST/Target/GTO/HR8799/output_with_photom/obs18/jw01194018001_02101_00001_mirimage_rate.fits', '/home/cperrot/Work/raw/JWST/Target/GTO/HR8799/output_with_photom/obs18/jw01194018001_02101_00002_mirimage_rate.fits']).
2023-11-16 15:57:51,435 - stpipe.Image2Pipeline.bkg_subtract - INFO - Step bkg_subtract parameters are: {'pre_hooks': [], 'post_hooks': [], 'output_file': None, 'output_dir': None, 'output_ext': '.fits', 'output_use_model': False, 'output_use_index': True, 'save_results': False, 'skip': False, 'suffix': 'bsub', 'search_output_file': True, 'input_dir': '', 'save_combined_background': False, 'sigma': 3.0, 'maxiters': None, 'wfss_mmag_extract': None}
2023-11-16 15:57:51,466 - stpipe.Image2Pipeline.bkg_subtract - INFO - Accumulate bkg from /home/cperrot/Work/raw/JWST/Target/G

2023-11-16 15:57:54,004 - stpipe.Image2Pipeline.assign_wcs - INFO - Created a MIRI mir_4qpm pipeline with references {'distortion': '/home/cperrot/crds_cache/references/jwst/miri/jwst_miri_distortion_0047.asdf', 'filteroffset': '/home/cperrot/crds_cache/references/jwst/miri/jwst_miri_filteroffset_0007.asdf', 'specwcs': None, 'regions': None, 'wavelengthrange': None, 'camera': None, 'collimator': None, 'disperser': None, 'fore': None, 'fpa': None, 'msa': None, 'ote': None, 'ifupost': None, 'ifufore': None, 'ifuslicer': None}
2023-11-16 15:57:54,117 - stpipe.Image2Pipeline.assign_wcs - INFO - Update S_REGION to POLYGON ICRS  346.631242925 19.907177241 346.638565200 19.907450097 346.638281968 19.916244760 346.630932735 19.915988852
2023-11-16 15:57:54,118 - stpipe.Image2Pipeline.assign_wcs - INFO - assign_wcs updated S_REGION to POLYGON ICRS  346.631242925 19.907177241 346.638565200 19.907450097 346.638281968 19.916244760 346.630932735 19.915988852
2023-11-16 15:57:54,118 - stpipe.Image2P

2023-11-16 15:57:55,342 - stpipe.Image2Pipeline.photom - INFO - Step photom parameters are: {'pre_hooks': [], 'post_hooks': [], 'output_file': None, 'output_dir': None, 'output_ext': '.fits', 'output_use_model': False, 'output_use_index': True, 'save_results': False, 'skip': False, 'suffix': 'photom', 'search_output_file': True, 'input_dir': '', 'inverse': False, 'source_type': None, 'mrs_time_correction': True}
2023-11-16 15:57:55,373 - stpipe.Image2Pipeline.photom - INFO - Using photom reference file: /home/cperrot/crds_cache/references/jwst/miri/jwst_miri_photom_0173.fits
2023-11-16 15:57:55,374 - stpipe.Image2Pipeline.photom - INFO - Using area reference file: N/A
2023-11-16 15:57:55,463 - stpipe.Image2Pipeline.photom - INFO - Using instrument: MIRI
2023-11-16 15:57:55,464 - stpipe.Image2Pipeline.photom - INFO -  detector: MIRIMAGE
2023-11-16 15:57:55,465 - stpipe.Image2Pipeline.photom - INFO -  exp_type: MIR_4QPM
2023-11-16 15:57:55,465 - stpipe.Image2Pipeline.photom - INFO -  fil

...

2023-11-16 15:58:00,821 - stpipe.Image2Pipeline.bkg_subtract - INFO - Accumulate bkg from /home/cperrot/Work/raw/JWST/Target/GTO/HR8799/output_with_photom/obs18/jw01194018001_02101_00002_mirimage_rate.fits
2023-11-16 15:58:00,866 - stpipe.Image2Pipeline.bkg_subtract - WARNING - /home/cperrot/anaconda3/envs/mirisim/lib/python3.9/site-packages/astropy/stats/sigma_clipping.py:364: AstropyUserWarning: Input data contains invalid values (NaNs or infs), which were automatically clipped.
  warnings.warn(

2023-11-16 15:58:00,907 - stpipe.Image2Pipeline.bkg_subtract - INFO - Step bkg_subtract done
2023-11-16 15:58:00,983 - stpipe.Image2Pipeline.assign_wcs - INFO - Step assign_wcs running with args (<ImageModel(224, 288) from jw01194016001_04101_00009_mirimage_rate.fits>,).
2023-11-16 15:58:00,985 - stpipe.Image2Pipeline.assign_wcs - INFO - Step assign_wcs parameters are: {'pre_hooks': [], 'post_hooks': [], 'output_file': None, 'output_dir': None, 'output_ext': '.fits', 'output_use_model': Fals

2023-11-16 15:58:03,114 - stpipe.Image2Pipeline - INFO - Step Image2Pipeline running with args ('HD218261_lvl2_asn.json',).
2023-11-16 15:58:03,119 - stpipe.Image2Pipeline - INFO - Step Image2Pipeline parameters are: {'pre_hooks': [], 'post_hooks': [], 'output_file': None, 'output_dir': '/home/cperrot/Work/raw/JWST/Target/GTO/HR8799/output_with_photom/obs16/TA/', 'output_ext': '.fits', 'output_use_model': False, 'output_use_index': True, 'save_results': True, 'skip': False, 'suffix': None, 'search_output_file': True, 'input_dir': '', 'save_bsub': False, 'steps': {'bkg_subtract': {'pre_hooks': [], 'post_hooks': [], 'output_file': None, 'output_dir': None, 'output_ext': '.fits', 'output_use_model': False, 'output_use_index': True, 'save_results': False, 'skip': False, 'suffix': None, 'search_output_file': True, 'input_dir': '', 'save_combined_background': False, 'sigma': 3.0, 'maxiters': None, 'wfss_mmag_extract': None}, 'assign_wcs': {'pre_hooks': [], 'post_hooks': [], 'output_file': No

{
    "asn_type": "image2",
    "asn_rule": "DMSLevel2bBase",
    "version_id": null,
    "code_version": "1.12.5",
    "degraded_status": "No known degraded exposures in association.",
    "program": "noprogram",
    "constraints": "No constraints",
    "asn_id": "a3001",
    "asn_pool": "HD218261_pool",
    "products": [
        {
            "name": "HD218261_dither1",
            "members": [
                {
                    "expname": "/home/cperrot/Work/raw/JWST/Target/GTO/HR8799/output_with_photom/obs16/TA/jw01194016001_02101_00001_mirimage_rate.fits",
                    "exptype": "science"
                }
            ]
        },
        {
            "name": "HD218261_dither2",
            "members": [
                {
                    "expname": "/home/cperrot/Work/raw/JWST/Target/GTO/HR8799/output_with_photom/obs16/TA/jw01194016001_02101_00002_mirimage_rate.fits",
                    "exptype": "science"
                }
            ]
        }
    ]
}


2023-11-16 15:58:03,338 - stpipe.Image2Pipeline.assign_wcs - INFO - Step assign_wcs running with args (<ImageModel(224, 288) from jw01194016001_02101_00001_mirimage_rate.fits>,).
2023-11-16 15:58:03,340 - stpipe.Image2Pipeline.assign_wcs - INFO - Step assign_wcs parameters are: {'pre_hooks': [], 'post_hooks': [], 'output_file': None, 'output_dir': None, 'output_ext': '.fits', 'output_use_model': False, 'output_use_index': True, 'save_results': False, 'skip': False, 'suffix': None, 'search_output_file': True, 'input_dir': '', 'sip_approx': True, 'sip_max_pix_error': 0.1, 'sip_degree': None, 'sip_max_inv_pix_error': 0.1, 'sip_inv_degree': None, 'sip_npoints': 12, 'slit_y_low': -0.55, 'slit_y_high': 0.55}
2023-11-16 15:58:03,521 - stpipe.Image2Pipeline.assign_wcs - INFO - Created a MIRI mir_tacq pipeline with references {'distortion': '/home/cperrot/crds_cache/references/jwst/miri/jwst_miri_distortion_0047.asdf', 'filteroffset': '/home/cperrot/crds_cache/references/jwst/miri/jwst_miri_fil

2023-11-16 15:58:04,726 - stpipe.Image2Pipeline.photom - INFO -  exp_type: MIR_TACQ
2023-11-16 15:58:04,726 - stpipe.Image2Pipeline.photom - INFO -  filter: FND
2023-11-16 15:58:04,762 - stpipe.Image2Pipeline.photom - INFO -  subarray: MASK1550
2023-11-16 15:58:04,763 - stpipe.Image2Pipeline.photom - WARNING - Expected to find one matching row in table, found 0.
2023-11-16 15:58:04,763 - stpipe.Image2Pipeline.photom - WARNING - Expected to find one matching row in table, found 0.
2023-11-16 15:58:04,772 - stpipe.Image2Pipeline.photom - INFO - Step photom done
2023-11-16 15:58:04,773 - stpipe.Image2Pipeline - INFO - Finished processing product HD218261_dither2
2023-11-16 15:58:04,776 - stpipe.Image2Pipeline - INFO - ... ending calwebb_image2
2023-11-16 15:58:04,777 - stpipe.Image2Pipeline - INFO - Results used CRDS context: jwst_1148.pmap
2023-11-16 15:58:04,777 - stpipe.Image2Pipeline - INFO - Results used CRDS context: jwst_1148.pmap
2023-11-16 15:58:04,935 - stpipe.Image2Pipeline - I

In [5]:
#Define path
path = "/home/cperrot/Work/raw/JWST/Target/GO/P3662/output_with_photom/"
output_dir = path
name_pl = 'HD203010'
filter_name = ["F1140C"] #["F1065C","F1140C","F1550C"]#, "F2100W"]
cmaps = ["Blues_r","Greens_r","Oranges_r"]

#Number of the observations : useful to retrieve the data 
#For each filter : science obs, background, ref, and ref background
num_obs = [[9,10]] 

D = 6.57 #Diameter of the telescope
wavelength_filter = [11.40]
nb_filter = len(wavelength_filter)
fwhm = [((wavelength_filter[t]*10**-6)/6.57)*(180*3600/np.pi) for t in range(nb_filter)]
pix_size = 0.11 #arcsec/pixel

#Center corono : DEC, RA 
yxc1065 = [112.00, 120.25]  
yxc1140 = [112.16, 119.77]  
yxc1550 = [113.20, 119.73] 
yxc2330 = [167.88, 144.33]  
center_corono = (yxc1065,yxc1140,yxc1550)

In [6]:
for ft in range(len(num_obs)):

    #Retrieve from MAST : 
    files_rate = sorted(glob.glob(path+'obs'+str(num_obs[ft][0])+'*/*_mirimage_rate.fits'))
    files_bkg_rate = sorted(glob.glob(path+'obs'+str(num_obs[ft][1])+'*/*_mirimage_rate.fits'))
    #files_ref_rate = sorted(glob.glob(path+'obs'+str(num_obs[ft][2])+'*/*_mirimage_rate.fits'))
    #files_ref_bkg_rate = sorted(glob.glob(path+'obs'+str(num_obs[ft][3])+'*/*_mirimage_rate.fits'))

    files_TA = sorted(glob.glob(path+'obs'+str(num_obs[ft][0])+'*/TA/*02101*_mirimage_rate.fits'))
       
    #Liste ASN with science and bkg files : 
    asnlist = writeasnfile(files_rate, files_bkg_rate, name_pl)
    #Run the pipeline : 
    image2(asnlist, output_dir+"obs"+str(num_obs[ft][0])+"/")

    #Same for ref files
    #asnlist_ref = writeasnfile(files_ref_rate, files_ref_bkg_rate,name_pl)
    #image2(asnlist_ref,output_dir+"obs"+str(num_obs[ft][2])+"/")
    
    #Same for TA (useful for photometry only)
    asnlist2 = writeasnfile2(files_TA,name_pl)
    image2(asnlist2, output_dir+"obs"+str(num_obs[ft][0])+"/TA/")

2023-11-16 15:58:05,099 - stpipe - WARNING - /home/cperrot/anaconda3/envs/mirisim/lib/python3.9/site-packages/jwst/associations/association.py:215: UserWarning: 'expname' contains path, but should only be a filename.  All input files should be in a single directory, so no path is needed.
  warnings.warn(err_str, UserWarning)

2023-11-16 15:58:05,105 - stpipe.Image2Pipeline - INFO - Image2Pipeline instance created.
2023-11-16 15:58:05,107 - stpipe.Image2Pipeline.bkg_subtract - INFO - BackgroundStep instance created.
2023-11-16 15:58:05,110 - stpipe.Image2Pipeline.assign_wcs - INFO - AssignWcsStep instance created.
2023-11-16 15:58:05,111 - stpipe.Image2Pipeline.flat_field - INFO - FlatFieldStep instance created.
2023-11-16 15:58:05,112 - stpipe.Image2Pipeline.photom - INFO - PhotomStep instance created.
2023-11-16 15:58:05,114 - stpipe.Image2Pipeline.resample - INFO - ResampleStep instance created.
2023-11-16 15:58:05,206 - stpipe.Image2Pipeline - INFO - Step Image2Pipeline running with

{
    "asn_type": "image2",
    "asn_rule": "DMSLevel2bBase",
    "version_id": null,
    "code_version": "1.12.5",
    "degraded_status": "No known degraded exposures in association.",
    "program": "noprogram",
    "constraints": "No constraints",
    "asn_id": "a3001",
    "asn_pool": "HD203010_pool",
    "products": [
        {
            "name": "HD203010_dither1",
            "members": [
                {
                    "expname": "/home/cperrot/Work/raw/JWST/Target/GO/P3662/output_with_photom/obs9/jw03662009001_04101_00001_mirimage_rate.fits",
                    "exptype": "science"
                },
                {
                    "expname": "/home/cperrot/Work/raw/JWST/Target/GO/P3662/output_with_photom/obs10/jw03662010001_02101_00001_mirimage_rate.fits",
                    "exptype": "background"
                },
                {
                    "expname": "/home/cperrot/Work/raw/JWST/Target/GO/P3662/output_with_photom/obs10/jw03662010001_02101_00002_m

2023-11-16 15:58:05,361 - stpipe.Image2Pipeline.bkg_subtract - INFO - Step bkg_subtract running with args (<ImageModel(224, 288) from jw03662009001_04101_00001_mirimage_rate.fits>, ['/home/cperrot/Work/raw/JWST/Target/GO/P3662/output_with_photom/obs10/jw03662010001_02101_00001_mirimage_rate.fits', '/home/cperrot/Work/raw/JWST/Target/GO/P3662/output_with_photom/obs10/jw03662010001_02101_00002_mirimage_rate.fits']).
2023-11-16 15:58:05,363 - stpipe.Image2Pipeline.bkg_subtract - INFO - Step bkg_subtract parameters are: {'pre_hooks': [], 'post_hooks': [], 'output_file': None, 'output_dir': None, 'output_ext': '.fits', 'output_use_model': False, 'output_use_index': True, 'save_results': False, 'skip': False, 'suffix': 'bsub', 'search_output_file': True, 'input_dir': '', 'save_combined_background': False, 'sigma': 3.0, 'maxiters': None, 'wfss_mmag_extract': None}
2023-11-16 15:58:05,397 - stpipe.Image2Pipeline.bkg_subtract - INFO - Accumulate bkg from /home/cperrot/Work/raw/JWST/Target/GO/P3

2023-11-16 15:58:06,945 - stpipe.Image2Pipeline.assign_wcs - INFO - Created a MIRI mir_4qpm pipeline with references {'distortion': '/home/cperrot/crds_cache/references/jwst/miri/jwst_miri_distortion_0047.asdf', 'filteroffset': '/home/cperrot/crds_cache/references/jwst/miri/jwst_miri_filteroffset_0010.asdf', 'specwcs': None, 'regions': None, 'wavelengthrange': None, 'camera': None, 'collimator': None, 'disperser': None, 'fore': None, 'fpa': None, 'msa': None, 'ote': None, 'ifupost': None, 'ifufore': None, 'ifuslicer': None}
2023-11-16 15:58:07,015 - stpipe.Image2Pipeline.assign_wcs - INFO - Update S_REGION to POLYGON ICRS  320.320588607 -49.944164864 320.326838805 -49.938602817 320.315768476 -49.933410674 320.309477521 -49.938996441
2023-11-16 15:58:07,016 - stpipe.Image2Pipeline.assign_wcs - INFO - assign_wcs updated S_REGION to POLYGON ICRS  320.320588607 -49.944164864 320.326838805 -49.938602817 320.315768476 -49.933410674 320.309477521 -49.938996441
2023-11-16 15:58:07,016 - stpipe

2023-11-16 15:58:08,277 - stpipe.Image2Pipeline.photom - INFO - Step photom parameters are: {'pre_hooks': [], 'post_hooks': [], 'output_file': None, 'output_dir': None, 'output_ext': '.fits', 'output_use_model': False, 'output_use_index': True, 'save_results': False, 'skip': False, 'suffix': 'photom', 'search_output_file': True, 'input_dir': '', 'inverse': False, 'source_type': None, 'mrs_time_correction': True}
2023-11-16 15:58:08,306 - stpipe.Image2Pipeline.photom - INFO - Using photom reference file: /home/cperrot/crds_cache/references/jwst/miri/jwst_miri_photom_0172.fits
2023-11-16 15:58:08,307 - stpipe.Image2Pipeline.photom - INFO - Using area reference file: N/A
2023-11-16 15:58:08,382 - stpipe.Image2Pipeline.photom - INFO - Using instrument: MIRI
2023-11-16 15:58:08,383 - stpipe.Image2Pipeline.photom - INFO -  detector: MIRIMAGE
2023-11-16 15:58:08,384 - stpipe.Image2Pipeline.photom - INFO -  exp_type: MIR_4QPM
2023-11-16 15:58:08,384 - stpipe.Image2Pipeline.photom - INFO -  fil

...

2023-11-16 15:58:12,494 - stpipe.Image2Pipeline.photom - INFO - Using area reference file: N/A
2023-11-16 15:58:12,596 - stpipe.Image2Pipeline.photom - INFO - Using instrument: MIRI
2023-11-16 15:58:12,597 - stpipe.Image2Pipeline.photom - INFO -  detector: MIRIMAGE
2023-11-16 15:58:12,598 - stpipe.Image2Pipeline.photom - INFO -  exp_type: MIR_4QPM
2023-11-16 15:58:12,598 - stpipe.Image2Pipeline.photom - INFO -  filter: F1140C
2023-11-16 15:58:12,640 - stpipe.Image2Pipeline.photom - INFO -  subarray: MASK1140
2023-11-16 15:58:12,641 - stpipe.Image2Pipeline.photom - INFO - PHOTMJSR value: 2.82222
2023-11-16 15:58:12,675 - stpipe.Image2Pipeline.photom - INFO - Step photom done
2023-11-16 15:58:12,676 - stpipe.Image2Pipeline - INFO - Finished processing product HD203010_dither7
2023-11-16 15:58:12,679 - stpipe.Image2Pipeline - INFO - Processing product HD203010_dither8
2023-11-16 15:58:12,679 - stpipe.Image2Pipeline - INFO - Working on input /home/cperrot/Work/raw/JWST/Target/GO/P3662/outp

2023-11-16 15:58:14,203 - stpipe.Image2Pipeline.bkg_subtract - INFO - Accumulate bkg from /home/cperrot/Work/raw/JWST/Target/GO/P3662/output_with_photom/obs10/jw03662010001_02101_00002_mirimage_rate.fits
2023-11-16 15:58:14,257 - stpipe.Image2Pipeline.bkg_subtract - WARNING - /home/cperrot/anaconda3/envs/mirisim/lib/python3.9/site-packages/astropy/stats/sigma_clipping.py:364: AstropyUserWarning: Input data contains invalid values (NaNs or infs), which were automatically clipped.
  warnings.warn(

2023-11-16 15:58:14,302 - stpipe.Image2Pipeline.bkg_subtract - INFO - Step bkg_subtract done
2023-11-16 15:58:14,390 - stpipe.Image2Pipeline.assign_wcs - INFO - Step assign_wcs running with args (<ImageModel(224, 288) from jw03662009001_04101_00009_mirimage_rate.fits>,).
2023-11-16 15:58:14,392 - stpipe.Image2Pipeline.assign_wcs - INFO - Step assign_wcs parameters are: {'pre_hooks': [], 'post_hooks': [], 'output_file': None, 'output_dir': None, 'output_ext': '.fits', 'output_use_model': False,

2023-11-16 15:58:16,582 - stpipe.Image2Pipeline - INFO - Step Image2Pipeline running with args ('HD203010_lvl2_asn.json',).
2023-11-16 15:58:16,590 - stpipe.Image2Pipeline - INFO - Step Image2Pipeline parameters are: {'pre_hooks': [], 'post_hooks': [], 'output_file': None, 'output_dir': '/home/cperrot/Work/raw/JWST/Target/GO/P3662/output_with_photom/obs9/TA/', 'output_ext': '.fits', 'output_use_model': False, 'output_use_index': True, 'save_results': True, 'skip': False, 'suffix': None, 'search_output_file': True, 'input_dir': '', 'save_bsub': False, 'steps': {'bkg_subtract': {'pre_hooks': [], 'post_hooks': [], 'output_file': None, 'output_dir': None, 'output_ext': '.fits', 'output_use_model': False, 'output_use_index': True, 'save_results': False, 'skip': False, 'suffix': None, 'search_output_file': True, 'input_dir': '', 'save_combined_background': False, 'sigma': 3.0, 'maxiters': None, 'wfss_mmag_extract': None}, 'assign_wcs': {'pre_hooks': [], 'post_hooks': [], 'output_file': None,

{
    "asn_type": "image2",
    "asn_rule": "DMSLevel2bBase",
    "version_id": null,
    "code_version": "1.12.5",
    "degraded_status": "No known degraded exposures in association.",
    "program": "noprogram",
    "constraints": "No constraints",
    "asn_id": "a3001",
    "asn_pool": "HD203010_pool",
    "products": [
        {
            "name": "HD203010_dither1",
            "members": [
                {
                    "expname": "/home/cperrot/Work/raw/JWST/Target/GO/P3662/output_with_photom/obs9/TA/jw03662009001_02101_00002_mirimage_rate.fits",
                    "exptype": "science"
                }
            ]
        }
    ]
}


2023-11-16 15:58:16,678 - stpipe.Image2Pipeline - INFO - Prefetching reference files for dataset: 'jw03662009001_02101_00002_mirimage_rate.fits' reftypes = ['area', 'camera', 'collimator', 'disperser', 'distortion', 'drizpars', 'filteroffset', 'fore', 'fpa', 'ifufore', 'ifupost', 'ifuslicer', 'msa', 'ote', 'photom', 'regions', 'specwcs', 'wavelengthrange', 'wfssbkg']
2023-11-16 15:58:16,682 - stpipe.Image2Pipeline - INFO - Prefetch for AREA reference file is 'N/A'.
2023-11-16 15:58:16,682 - stpipe.Image2Pipeline - INFO - Prefetch for CAMERA reference file is 'N/A'.
2023-11-16 15:58:16,683 - stpipe.Image2Pipeline - INFO - Prefetch for COLLIMATOR reference file is 'N/A'.
2023-11-16 15:58:16,683 - stpipe.Image2Pipeline - INFO - Prefetch for DISPERSER reference file is 'N/A'.
2023-11-16 15:58:16,685 - stpipe.Image2Pipeline - INFO - Prefetch for DISTORTION reference file is '/home/cperrot/crds_cache/references/jwst/miri/jwst_miri_distortion_0047.asdf'.
2023-11-16 15:58:16,685 - stpipe.Image